In [ ]:
!pip install zoobot[pytorch_colab]

In [ ]:
from google.colab import drive
drive.mount('/content/mydrive',force_remount=True)

In [ ]:
import os
print(os.listdir('/content/mydrive/MyDrive/'))

In [ ]:
import zipfile
import os

# Create destination folders
os.makedirs('/content/mydrive/MyDrive/galaxy_zoo/images_test_rev1/', exist_ok=True)
os.makedirs('/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/', exist_ok=True)

# Unzip training set
print("Unzipping training set...")
with zipfile.ZipFile('/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1.zip', 'r') as z:
    z.extractall('/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/')
print("Done! Training set ready.")

# Unzip test set
print("Unzipping test set...")
with zipfile.ZipFile('/content/mydrive/MyDrive/galaxy_zoo/images_test_rev1.zip', 'r') as z:
    z.extractall('/content/mydrive/MyDrive/galaxy_zoo/images_test_rev1/')
print("Done! Test set ready.")

In [ ]:
training_count = len(os.listdir('/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/'))
test_count = len(os.listdir('/content/mydrive/MyDrive/galaxy_zoo/images_test_rev1/images_test_rev1/'))

print(f"Training images: {training_count}")
print(f"Test images: {test_count}")

In [ ]:
!pip install timm

In [ ]:
import pandas as pd
import numpy as np
import torch
from zoobot.pytorch.training import finetune
from zoobot.pytorch.predictions import predict_on_catalog


# Load ground truth labels
solutions = pd.read_csv('/content/mydrive/MyDrive/galaxy_zoo/training_solutions_rev1.csv')
print(solutions.shape)      # should be (61578, 38) — GalaxyID + 37 vote fractions
print(solutions.head())

In [ ]:
import os
import glob
import pandas as pd

# Get all training image paths
image_paths = glob.glob('/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/*.jpg')

# Extract galaxy IDs from filenames (filename is GalaxyID.jpg)
image_df = pd.DataFrame({
    'file_loc': image_paths,
    'GalaxyID': [int(os.path.basename(p).replace('.jpg', '')) for p in image_paths]
})

# Merge with solutions to get ground truth
merged = image_df.merge(solutions, on='GalaxyID')
print(f"Matched {len(merged)} images with ground truth labels")

In [ ]:
sample = merged.sample(n=500, random_state=42).reset_index(drop=True)

# Save just the file paths for Zoobot
predict_df = sample[['file_loc']].copy()

In [ ]:
from zoobot.pytorch.training import finetune
from zoobot.shared import schemas

model = finetune.FinetuneableZoobotTree(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    schema=schemas.gz2_ortho_schema
)
model.eval()

# Check what label cols it uses
print(schemas.gz2_ortho_schema.label_cols)

In [ ]:
# Exact mapping: Zoobot gz2_ortho_schema → Kaggle column names
zoobot_to_kaggle = {
    'smooth-or-featured-gz2_smooth':             'Class1.1',
    'smooth-or-featured-gz2_featured-or-disk':   'Class1.2',
    'smooth-or-featured-gz2_artifact':            'Class1.3',
    'disk-edge-on-gz2_yes':                       'Class2.1',
    'disk-edge-on-gz2_no':                        'Class2.2',
    'has-spiral-arms-gz2_yes':                    'Class4.1',
    'has-spiral-arms-gz2_no':                     'Class4.2',
    'bar-gz2_yes':                                'Class5.1',
    'bar-gz2_no':                                 'Class5.2',
    'bulge-size-gz2_dominant':                    'Class6.1',
    'bulge-size-gz2_obvious':                     'Class6.2',
    'bulge-size-gz2_just-noticeable':             'Class6.3',
    'bulge-size-gz2_no':                          'Class6.4',
    'something-odd-gz2_yes':                      'Class8.1',
    'something-odd-gz2_no':                       'Class8.2',
    'how-rounded-gz2_round':                      'Class7.1',
    'how-rounded-gz2_in-between':                 'Class7.2',
    'how-rounded-gz2_cigar':                      'Class7.3',
    'bulge-shape-gz2_round':                      'Class9.1',
    'bulge-shape-gz2_boxy':                       'Class9.2',
    'bulge-shape-gz2_no-bulge':                   'Class9.3',
    'spiral-winding-gz2_tight':                   'Class10.1',
    'spiral-winding-gz2_medium':                  'Class10.2',
    'spiral-winding-gz2_loose':                   'Class10.3',
    'spiral-arm-count-gz2_1':                     'Class11.1',
    'spiral-arm-count-gz2_2':                     'Class11.2',
    'spiral-arm-count-gz2_3':                     'Class11.3',
    'spiral-arm-count-gz2_4':                     'Class11.4',
    'spiral-arm-count-gz2_more-than-4':           'Class11.5',
    'spiral-arm-count-gz2_cant-tell':             'Class11.6',
}

# The 7 Kaggle columns Zoobot doesn't cover
missing_kaggle_cols = ['Class3.1', 'Class3.2', 'Class5.3', 'Class5.4', 'Class8.3',
                       'Class8.4', 'Class8.5', 'Class8.6', 'Class8.7']

print(f"Zoobot covers: {len(zoobot_to_kaggle)} columns")
print(f"Missing from Zoobot: {missing_kaggle_cols}")

In [ ]:
# Add the required id_str column
predict_df = sample[['file_loc']].copy()
predict_df['id_str'] = predict_df['file_loc'].apply(lambda x: os.path.basename(x).replace('.jpg', ''))

print(f"Images ready: {len(predict_df)}")
print(predict_df.head())

In [ ]:
zoobot_to_kaggle = {
    # Question 1 — Smooth or Featured?
    'smooth-or-featured-gz2_smooth':            'Class1.1',
    'smooth-or-featured-gz2_featured-or-disk':  'Class1.2',
    'smooth-or-featured-gz2_artifact':          'Class1.3',

    # Question 2 — Edge on disk?
    'disk-edge-on-gz2_yes':                     'Class2.1',
    'disk-edge-on-gz2_no':                      'Class2.2',

    # Question 4 — Spiral arms?
    'has-spiral-arms-gz2_yes':                  'Class4.1',
    'has-spiral-arms-gz2_no':                   'Class4.2',

    # Question 5 — Bar?
    'bar-gz2_yes':                              'Class5.1',
    'bar-gz2_no':                               'Class5.2',

    # Question 6 — Bulge size?
    'bulge-size-gz2_dominant':                  'Class6.1',
    'bulge-size-gz2_obvious':                   'Class6.2',

    # Question 7 — How rounded?
    'how-rounded-gz2_round':                    'Class7.1',
    'how-rounded-gz2_in-between':               'Class7.2',
    'how-rounded-gz2_cigar':                    'Class7.3',

    # Question 8 — Something odd?
    'something-odd-gz2_yes':                    'Class8.1',
    'something-odd-gz2_no':                     'Class8.2',

    # Question 9 — Bulge shape?
    'bulge-shape-gz2_round':                    'Class9.1',
    'bulge-shape-gz2_boxy':                     'Class9.2',
    'bulge-shape-gz2_no-bulge':                 'Class9.3',

    # Question 10 — Spiral winding?
    'spiral-winding-gz2_tight':                 'Class10.1',
    'spiral-winding-gz2_medium':                'Class10.2',
    'spiral-winding-gz2_loose':                 'Class10.3',

    # Question 11 — Spiral arm count?
    'spiral-arm-count-gz2_1':                   'Class11.1',
    'spiral-arm-count-gz2_2':                   'Class11.2',
    'spiral-arm-count-gz2_3':                   'Class11.3',
    'spiral-arm-count-gz2_4':                   'Class11.4',
    'spiral-arm-count-gz2_more-than-4':         'Class11.5',
    'spiral-arm-count-gz2_cant-tell':           'Class11.6',
}

# Kaggle columns NOT covered by Zoobot
not_covered = ['Class3.1', 'Class3.2',           # edge-on bulge shape
               'Class5.3', 'Class5.4',           # bar detail
               'Class6.3', 'Class6.4',           # bulge size detail
               'Class8.3', 'Class8.4',
               'Class8.5', 'Class8.6', 'Class8.7']  # odd feature types

matched_kaggle_cols = list(zoobot_to_kaggle.values())
print(f"Matched columns: {len(matched_kaggle_cols)}")
print(f"Not covered by Zoobot: {not_covered}")

In [ ]:
print(solutions.columns.tolist())

In [ ]:
# Check exact Zoobot output columns
predictions_v2 = pd.read_csv('/content/zoobot_predictions_v2.csv')
print(predictions_v2.columns.tolist())


In [ ]:
from PIL import Image
import numpy as np

# Compute mean and std from a sample of your images
sample_paths = sample['file_loc'].tolist()[:200]

pixel_sum = np.zeros(3)
pixel_sq_sum = np.zeros(3)
pixel_count = 0

for path in sample_paths:
    img = np.array(Image.open(path).convert('RGB').crop(
        (108, 108, 315, 315)  # center crop 207x207 from 424x424
    )) / 255.0

    pixel_sum += img.mean(axis=(0,1))
    pixel_sq_sum += (img**2).mean(axis=(0,1))
    pixel_count += 1

mean = pixel_sum / pixel_count
std = np.sqrt(pixel_sq_sum / pixel_count - mean**2)

print(f"Computed mean: {mean.round(4).tolist()}")
print(f"Computed std:  {std.round(4).tolist()}")

In [ ]:
from PIL import Image
import numpy as np

# Compute mean and std from a sample of your images
sample_paths = sample['file_loc'].tolist()[:200]

pixel_sum = np.zeros(3)
pixel_sq_sum = np.zeros(3)
pixel_count = 0

for path in sample_paths:
    img = np.array(Image.open(path).convert('RGB').crop(
        (87, 87, 337, 337)# center crop 207x207 from 424x424
    )) / 255.0

    pixel_sum += img.mean(axis=(0,1))
    pixel_sq_sum += (img**2).mean(axis=(0,1))
    pixel_count += 1

mean = pixel_sum / pixel_count
std = np.sqrt(pixel_sq_sum / pixel_count - mean**2)

print(f"Computed mean: {mean.round(4).tolist()}")
print(f"Computed std:  {std.round(4).tolist()}")

In [ ]:
import torchvision.transforms.v2 as transforms_v2
from PIL import Image
from zoobot.pytorch.predictions import predict_on_catalog
from zoobot.pytorch.training import finetune
import torch
from zoobot.shared import schemas

inference_transform = transforms_v2.Compose([
    transforms_v2.CenterCrop(250),       # remove black border first
    transforms_v2.Resize((224, 224)),    # then resize to model input
    transforms_v2.ToImage(),
    transforms_v2.ToDtype(torch.float32, scale=True),
    transforms_v2.Normalize(
        mean=[0.0843, 0.0734, 0.0573],
        std=[0.1336, 0.1122, 0.0963]
    )
])

# See exactly what output columns GZ2 schema produces
print(schemas.gz2_ortho_schema.label_cols)



In [ ]:
predict_on_catalog.predict(
    predict_df,
    model,
    label_cols=schemas.gz2_ortho_schema.label_cols,
    save_loc='/content/zoobot_predictions_v2.csv',
    inference_transform=inference_transform
)
predictions = pd.read_csv('/content/zoobot_predictions_v2.csv')
print(predictions.head())

In [ ]:
# Check Zoobot output columns
predictions = pd.read_csv('/content/zoobot_predictions_v2.csv')
print("Zoobot output columns:")
print(predictions.columns.tolist())

# Check Kaggle solution columns
print("\nKaggle solution columns:")
print(solutions.columns.tolist())

In [ ]:
import inspect
print(inspect.signature(predict_on_catalog.predict))

In [ ]:
# Step 1 — Load raw predictions from CSV
predictions_v2 = pd.read_csv('/content/zoobot_predictions_v2.csv')

# Step 2 — Rename Zoobot column names to Kaggle column names
predictions_renamed = predictions_v2.rename(columns=zoobot_to_kaggle)

print(predictions_renamed.columns.tolist())
print(predictions_renamed.shape)

In [ ]:
# Zoobot outputs raw vote counts per question group
# We need to normalize within each question so they sum to 1

# Define question groups — columns belonging to same question must sum to 1
question_groups = [
    ['Class1.1', 'Class1.2', 'Class1.3'],
    ['Class2.1', 'Class2.2'],
    ['Class4.1', 'Class4.2'],
    ['Class5.1', 'Class5.2'],
    ['Class6.1', 'Class6.2'],
    ['Class7.1', 'Class7.2', 'Class7.3'],
    ['Class8.1', 'Class8.2'],
    ['Class9.1', 'Class9.2', 'Class9.3'],
    ['Class10.1', 'Class10.2', 'Class10.3'],
    ['Class11.1', 'Class11.2', 'Class11.3', 'Class11.4', 'Class11.5', 'Class11.6'],
]

# Normalize each question group so votes sum to 1
predictions_normalized = predictions_renamed.copy()
for group in question_groups:
    # Only normalize columns that exist in our matched set
    group_cols = [c for c in group if c in matched_kaggle_cols]
    if group_cols:
        row_sums = predictions_normalized[group_cols].sum(axis=1)
        predictions_normalized[group_cols] = predictions_normalized[group_cols].div(row_sums, axis=0)

# Verify normalization worked — each group should now sum to ~1
print("Sample normalized predictions (first 3 rows):")
print(predictions_normalized[matched_kaggle_cols].head(3).round(4))

print("\nNormalized value ranges (should be 0-1):")
print(predictions_normalized[matched_kaggle_cols].describe().round(4))

In [ ]:
ground_truth = sample[matched_kaggle_cols].reset_index(drop=True).values
predicted = predictions_normalized[matched_kaggle_cols].values

rmse = np.sqrt(np.mean((predicted - ground_truth) ** 2))
print(f"\nZoobot RMSE (normalized, 28 matched columns): {rmse:.5f}")
print(f"Dieleman winner (all 37):                     0.07492")
print(f"Central pixel baseline:                       0.16194")
print(f"Decent range:                                 0.10 - 0.12")

In [ ]:
print("Sample predictions (first 3 rows):")
print(predictions_renamed[matched_kaggle_cols].head(3))

print("\nPrediction value ranges:")
print(predictions_renamed[matched_kaggle_cols].describe())

print("\nGround truth value ranges:")
print(sample[matched_kaggle_cols].describe())

In [ ]:
# Check 1: Verify what your images actually look like after transform
from PIL import Image
import torchvision.transforms.v2 as transforms_v2
import torch
import matplotlib.pyplot as plt

# Load one image and apply transform
img = Image.open(sample['file_loc'].iloc[0])
print(f"Original image size: {img.size}")
print(f"Original image mode: {img.mode}")

# Apply transform step by step
crop = transforms_v2.CenterCrop(207)
img_cropped = crop(img)
print(f"After crop: {img_cropped.size}")

resize = transforms_v2.Resize((224, 224))
img_resized = resize(img_cropped)
print(f"After resize: {img_resized.size}")

# Show both
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title(f'Original {img.size}')
axes[1].imshow(img_cropped)
axes[1].set_title(f'After crop 207x207')
axes[2].imshow(img_resized)
axes[2].set_title(f'After resize 224x224')
plt.show()

In [ ]:
# Check 1: Verify what your images actually look like after transform
from PIL import Image
import torchvision.transforms.v2 as transforms_v2
import torch
import matplotlib.pyplot as plt

# Load one image and apply transform
common_id = enhanced_sample['GalaxyID'].iloc[0]

# Successfully find the exact file paths using the ID mask
orig_path = sample[sample['GalaxyID'] == common_id]['file_loc'].values[0]
enh_path = enhanced_sample[enhanced_sample['GalaxyID'] == common_id]['file_loc'].values[0]

# FIX: Pass the string paths directly to PIL
img = Image.open(orig_path)
img_enhanced = Image.open(enh_path)
print(f"Original image size: {img.size}")
print(f"Original image mode: {img.mode}")

# Apply transform step by step
crop = transforms_v2.CenterCrop(250)
img_cropped = crop(img)
print(f"After crop: {img_cropped.size}")

resize = transforms_v2.Resize((224, 224))
img_resized = resize(img_cropped)
print(f"After resize: {img_resized.size}")

# Show both
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title(f'Original {img.size}')
axes[1].imshow(img_cropped)
axes[1].set_title(f'After crop 207x207')
axes[2].imshow(img_resized)
axes[2].set_title(f'After resize 224x224')
plt.show()

In [ ]:
# Check 2: Verify predictions are actually varying meaningfully
# If the model is confused, all predictions will be near uniform (0.33 for 3-way, 0.5 for 2-way)
print("Mean predictions per column:")
print(predictions_normalized[matched_kaggle_cols].mean().round(4))

print("\nMean ground truth per column:")
print(sample[matched_kaggle_cols].mean().round(4))

In [ ]:
print(inference_transform)

In [ ]:
print([x for x in dir(finetune) if 'zoo' in x.lower() or 'encoder' in x.lower()])

In [ ]:
from zoobot.pytorch.training import finetune
from zoobot.shared import schemas
import torch.nn as nn

# Load the full FinetuneableZoobotTree model with a schema
model = finetune.FinetuneableZoobotTree(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    schema=schemas.gz2_ortho_schema # Provide a valid schema
)
encoder = model.encoder # Extract the encoder backbone
encoder.eval() # Set the encoder to evaluation mode

print("Encoder loaded and set to eval mode!")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as transforms_v2
from PIL import Image
from zoobot.pytorch.predictions import predict_on_catalog
from zoobot.pytorch.training import finetune
import torch
from zoobot.shared import schemas
# Step 2 — Define a simple dataset to load images
class GalaxyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df['file_loc'][idx]).convert('RGB')
        return self.transform(img)

dataset = GalaxyDataset(sample, inference_transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False)
print(f"Dataset ready: {len(dataset)} images")

In [ ]:
# Step 3 — Extract features from all images
features_list = []
with torch.no_grad():
    for batch in loader:
        feats = encoder(batch)
        features_list.append(feats.cpu().numpy())

features = np.concatenate(features_list, axis=0)
print(f"Features shape: {features.shape}")  # should be (500, 512) or similar

In [ ]:
# Step 4 — Train a simple regression head on Kaggle labels
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

# Split into train/val
X_train, X_val, y_train, y_val = train_test_split(
    features,
    sample[matched_kaggle_cols].values,
    test_size=0.2,
    random_state=42
)

# Train a Ridge regression — one per output column
from sklearn.multioutput import MultiOutputRegressor

regressor = MultiOutputRegressor(Ridge(alpha=1.0))
regressor.fit(X_train, y_train)
print("Regression head trained!")

In [ ]:
# Step 5 — Predict and calculate RMSE
y_pred = regressor.predict(X_val)

# Clip to 0-1 range
y_pred = np.clip(y_pred, 0, 1)

rmse = np.sqrt(np.mean((y_pred - y_val) ** 2))
print(f"\nZoobot + Ridge RMSE: {rmse:.5f}")
print(f"Dieleman winner:     0.07492")
print(f"Baseline:            0.16194")
print(f"Decent range:        0.10 - 0.12")

In [ ]:
import os
print(os.listdir('/content/mydrive/MyDrive/'))

In [ ]:
import glob
import os
import pandas as pd
import re

enhanced_paths = glob.glob('/content/mydrive/MyDrive/Galaxy_enhanced/**/*.jpg', recursive=True)

def extract_id(path):
    filename = os.path.basename(path)
    match = re.search(r'(\d+)', filename)
    return int(match.group(1)) if match else None

enhanced_df = pd.DataFrame({
    'file_loc': enhanced_paths,
    'GalaxyID': [extract_id(p) for p in enhanced_paths]
})

# Drop any rows where ID extraction failed
enhanced_df = enhanced_df.dropna(subset=['GalaxyID'])
enhanced_df['GalaxyID'] = enhanced_df['GalaxyID'].astype(int)

print(f"Total enhanced images found: {len(enhanced_df)}")
print(enhanced_df.tail())

In [ ]:
# Step 1 — Match enhanced images to the same GalaxyIDs used in original sample
enhanced_sample = enhanced_df[enhanced_df['GalaxyID'].isin(sample['GalaxyID'])].reset_index(drop=True)
print(f"Original sample size:        {len(sample)}")
print(f"Matched enhanced images:     {len(enhanced_sample)}")

# Check for any missing IDs
missing_ids = set(sample['GalaxyID']) - set(enhanced_sample['GalaxyID'])
print(f"GalaxyIDs in original but not in enhanced: {len(missing_ids)}")

In [ ]:
enhanced_sample.head()

In [ ]:
merged_2=enhanced_df.merge(solutions, on='GalaxyID')
print(f"Matched {len(merged_2)} images with ground truth labels")

sample_2=merged_2.sample(n=500, random_state=42).reset_index(drop=True)
predict_df_2=sample_2[['file_loc']].copy()
predict_df_2['id_str'] = predict_df_2['file_loc'].apply(lambda x: os.path.basename(x).replace('.jpg', ''))

print(f"Images ready: {len(predict_df_2)}")
print(sample_2.head())
print(predict_df_2.head())

In [ ]:
# Find duplicate GalaxyIDs
duplicates = enhanced_df[enhanced_df['GalaxyID'].duplicated(keep=False)]
print(f"Total duplicate rows: {len(duplicates)}")
#print(duplicates.sort_values('GalaxyID').head(20))

In [ ]:
enhanced_df = enhanced_df.drop_duplicates(subset='GalaxyID', keep='first').reset_index(drop=True)
print(f"After removing duplicates: {len(enhanced_df)} images")

In [ ]:
# Step 2 — Build predict_df for enhanced images
enhanced_predict_df = enhanced_sample[['file_loc']].copy()
enhanced_predict_df['id_str'] = enhanced_sample['GalaxyID'].astype(str)
print(enhanced_predict_df.head())

In [ ]:
enh_img = Image.open(enh_path).convert('RGB')
print(f"Enhanced image size: {enh_img.size}")
print(f"Original image size: {orig_img.size}")

enh_w, enh_h = enh_img.size

# Crop proportionally — same 49% center crop ratio as original
# Original: 207/424 = 0.489 of image width
crop_size = int(min(enh_w, enh_h) * 0.489)
print(f"Enhanced crop size should be: {crop_size}")

In [ ]:
# Compute mean and std from enhanced images
enhanced_sample_paths = enhanced_predict_df['file_loc'].tolist()[:200]

pixel_sum = np.zeros(3)
pixel_sq_sum = np.zeros(3)
pixel_count = 0

for path in enhanced_sample_paths:
    try:
        img = np.array(Image.open(path).convert('RGB').crop(
            (52, 52, 203, 203)
        )) / 255.0
        pixel_sum += img.mean(axis=(0,1))
        pixel_sq_sum += (img**2).mean(axis=(0,1))
        pixel_count += 1
    except:
        continue

enhanced_mean = pixel_sum / pixel_count
enhanced_std = np.sqrt(pixel_sq_sum / pixel_count - enhanced_mean**2)

print(f"Enhanced mean: {enhanced_mean.round(4).tolist()}")
print(f"Enhanced std:  {enhanced_std.round(4).tolist()}")

In [ ]:
enhanced_inference_transform = transforms_v2.Compose([
    transforms_v2.CenterCrop(125),   # proportional crop
    transforms_v2.Resize((224, 224)),
    transforms_v2.ToImage(),
    transforms_v2.ToDtype(torch.float32, scale=True),
    transforms_v2.Normalize(
        mean=[0.2558, 0.2638, 0.2426],   # from your enhanced image stats
        std=[0.2168, 0.2197, 0.2258]
    )
])

In [ ]:
# Check 1: Verify what your images actually look like after transform
from PIL import Image
import torchvision.transforms.v2 as transforms_v2
import torch
import matplotlib.pyplot as plt

# Load one image and apply transform
img = Image.open(enhanced_sample['file_loc'].iloc[0])
print(f"Original image size: {img.size}")
print(f"Original image mode: {img.mode}")

# Apply transform step by step
crop = transforms_v2.CenterCrop(151)
img_cropped = crop(img)
print(f"After crop: {img_cropped.size}")

resize = transforms_v2.Resize((224, 224))
img_resized = resize(img_cropped)
print(f"After resize: {img_resized.size}")

# Show both
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img)
axes[0].set_title(f'Original {img.size}')
axes[1].imshow(img_cropped)
axes[1].set_title(f'After crop 207x207')
axes[2].imshow(img_resized)
axes[2].set_title(f'After resize 224x224')
plt.show()

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

In [ ]:
# Step 3 — Extract features from enhanced images using same encoder
enhanced_dataset = GalaxyDataset(enhanced_predict_df, enhanced_inference_transform)
enhanced_loader = DataLoader(enhanced_dataset, batch_size=32, shuffle=False)
print(f"Enhanced dataset ready: {len(enhanced_dataset)} images")

enhanced_features_list = []
with torch.no_grad():
    for batch in enhanced_loader:
        feats = encoder(batch)
        enhanced_features_list.append(feats.cpu().numpy())

enhanced_features = np.concatenate(enhanced_features_list, axis=0)
print(f"Enhanced features shape: {enhanced_features.shape}")

In [ ]:
# Step 4 — Get ground truth for matched enhanced images
# Merge enhanced sample with solutions to get labels
enhanced_with_labels = enhanced_sample.merge(solutions, on='GalaxyID')
ground_truth_enhanced = enhanced_with_labels[matched_kaggle_cols].values
print(f"Ground truth shape: {ground_truth_enhanced.shape}")

In [ ]:
# Step 5 — Predict using the SAME regressor trained on original images
enhanced_pred = regressor.predict(enhanced_features)
enhanced_pred = np.clip(enhanced_pred, 0, 1)

# Calculate RMSE
rmse_enhanced = np.sqrt(np.mean((enhanced_pred - ground_truth_enhanced) ** 2))
print(f"\n=== RESULTS ===")
print(f"Original images RMSE:         {rmse:.5f}")
print(f"Enhanced images RMSE:         {rmse_enhanced:.5f}")
print(f"Difference:                   {rmse - rmse_enhanced:.5f}")
print(f"Dieleman winner:              0.07492")
print(f"Central pixel baseline:       0.16194")
if rmse_enhanced < rmse:
    print("\nEnhancement IMPROVED classification!")
elif rmse_enhanced > rmse:
    print("\nEnhancement DEGRADED classification.")
else:
    print("\nEnhancement had no effect.")

In [ ]:
enh_pred_df1 = pd.DataFrame(
    enhanced_pred,
    columns=matched_kaggle_cols
)
enh_pred_df1['GalaxyID'] = enhanced_sample['GalaxyID'].values
enh_pred_df1['file_loc'] = enhanced_sample['file_loc'].values

In [ ]:
import matplotlib.pyplot as plt

# Find a GalaxyID that exists in both
common_id = enhanced_sample['GalaxyID'].iloc[0]

orig_path = sample[sample['GalaxyID'] == common_id]['file_loc'].values[0]
enh_path = enhanced_sample[enhanced_sample['GalaxyID'] == common_id]['file_loc'].values[0]

orig_img = Image.open(orig_path).convert('RGB')
enh_img = Image.open(enh_path).convert('RGB')

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(orig_img)
axes[0].set_title(f'Original\nGalaxyID: {common_id}')
axes[1].imshow(enh_img)
axes[1].set_title(f'Enhanced\nGalaxyID: {common_id}')
plt.show()

# Also print pixel statistics
orig_arr = np.array(orig_img) / 255.0
enh_arr = np.array(enh_img) / 255.0
print(f"Original  — mean: {orig_arr.mean(axis=(0,1)).round(4)}, std: {orig_arr.std(axis=(0,1)).round(4)}")
print(f"Enhanced  — mean: {enh_arr.mean(axis=(0,1)).round(4)}, std: {enh_arr.std(axis=(0,1)).round(4)}")

In [ ]:
# Fine-tune the regressor on a small set of enhanced images
# Use 80% enhanced for training, 20% for testing
X_train_enh, X_val_enh, y_train_enh, y_val_enh = train_test_split(
    enhanced_features,
    ground_truth_enhanced,
    test_size=0.2,
    random_state=42
)

regressor_enh = MultiOutputRegressor(Ridge(alpha=5.0))
regressor_enh.fit(X_train_enh, y_train_enh)

y_pred_enh = np.clip(regressor_enh.predict(X_val_enh), 0, 1)
rmse_enh_finetuned = np.sqrt(np.mean((y_pred_enh - y_val_enh) ** 2))

print(f"Zoobot + Ridge on original images:              {rmse:.7f}")
print(f"Zoobot + Ridge on enhanced (cross-domain):      {rmse_enhanced:.7f}")
print(f"Zoobot + Ridge fine-tuned on enhanced images:   {rmse_enh_finetuned:.7f}")

In [ ]:
# Check if predictions are actually different between original and enhanced
orig_preds = np.clip(regressor.predict(X_val), 0, 1)
enh_preds = np.clip(regressor_enh.predict(X_val_enh), 0, 1)

print(f"Mean absolute difference between prediction sets: {np.mean(np.abs(orig_preds - enh_preds[:len(orig_preds)])):.5f}")

# Check prediction variance
print(f"Original pred std:  {orig_preds.std():.5f}")
print(f"Enhanced pred std:  {enh_preds.std():.5f}")
print(f"Ground truth std:   {y_val.std():.5f}")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor

# Try a small neural network head instead of Ridge
regressor_mlp = MultiOutputRegressor(
    MLPRegressor(
        hidden_layer_sizes=(256, 128),
        max_iter=500,
        random_state=42,
        early_stopping=True
    )
)

regressor_mlp.fit(X_train_enh, y_train_enh)
y_pred_mlp = np.clip(regressor_mlp.predict(X_val_enh), 0, 1)
rmse_mlp = np.sqrt(np.mean((y_pred_mlp - y_val_enh) ** 2))
print(f"MLP head RMSE on enhanced: {rmse_mlp:.5f}")
print(f"Ridge head RMSE on original: 0.11527")

In [ ]:
from sklearn.model_selection import cross_val_score

# Try different alpha values
for alpha in [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]:
    reg = MultiOutputRegressor(Ridge(alpha=alpha))
    reg.fit(X_train_enh, y_train_enh)
    pred = np.clip(reg.predict(X_val_enh), 0, 1)
    rmse = np.sqrt(np.mean((pred - y_val_enh) ** 2))
    print(f"Alpha={alpha}: RMSE={rmse:.5f}")

In [ ]:
# Train both heads with alpha=5.0
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

# Original images head
regressor_orig_5 = MultiOutputRegressor(Ridge(alpha=5.0))
regressor_orig_5.fit(X_train, y_train)
y_pred_orig_5 = np.clip(regressor_orig_5.predict(X_val), 0, 1)
rmse_orig_5 = np.sqrt(np.mean((y_pred_orig_5 - y_val) ** 2))

# Enhanced images headq
regressor_enh_5 = MultiOutputRegressor(Ridge(alpha=5.0))
regressor_enh_5.fit(X_train_enh, y_train_enh)
y_pred_enh_5 = np.clip(regressor_enh_5.predict(X_val_enh), 0, 1)
rmse_enh_5 = np.sqrt(np.mean((y_pred_enh_5 - y_val_enh) ** 2))

print(f"Alpha=5.0 Results:")
print(f"Original images RMSE:              {rmse_orig_5:.5f}")
print(f"Enhanced cross-domain RMSE:        0.28318")
print(f"Enhanced fine-tuned RMSE:          {rmse_enh_5:.5f}")
print(f"Dieleman winner:                   0.07492")
print(f"Central pixel baseline:            0.16194")

In [ ]:
all_pred_orig = np.clip(
    regressor_orig_5.predict(features),
    0,
    1
)
from sklearn.metrics import mean_squared_error
import numpy as np

orig_rmse_all = np.sqrt(mean_squared_error(
    ground_truth,
    all_pred_orig
))

print("RMSE:", orig_rmse_all)

In [ ]:
all_pred_enh = np.clip(
    regressor_enh_5.predict(enhanced_features),
    0,
    1
)
from sklearn.metrics import mean_squared_error
import numpy as np

rmse_all = np.sqrt(mean_squared_error(
    ground_truth_enhanced,
    all_pred_enh
))

print("RMSE:", rmse_all)


In [ ]:
final_pred_df_orig = pd.DataFrame(
    all_pred_orig,
    columns=matched_kaggle_cols
)

final_pred_df_orig['GalaxyID'] = enhanced_sample['GalaxyID'].values
final_pred_df_orig['file_loc'] = enhanced_sample['file_loc'].values

In [ ]:
final_pred_df = pd.DataFrame(
    all_pred_enh,
    columns=matched_kaggle_cols
)

final_pred_df['GalaxyID'] = enhanced_sample['GalaxyID'].values
final_pred_df['file_loc'] = enhanced_sample['file_loc'].values

In [ ]:
from skimage.exposure import match_histograms

def histogram_match_to_original(enhanced_path, original_path):
    """
    Match enhanced image histogram to its original counterpart
    Preserves structure, removes colour domain shift
    """
    enhanced = np.array(Image.open(enhanced_path).convert('RGB'))
    original = np.array(Image.open(original_path).convert('RGB'))

    # Resize enhanced to match original if different sizes
    if enhanced.shape != original.shape:
        enhanced = np.array(
            Image.fromarray(enhanced).resize(
                (original.shape[1], original.shape[0]),
                Image.LANCZOS
            )
        )

    # Match histogram channel by channel
    matched = match_histograms(enhanced, original, channel_axis=2)
    return matched.astype(np.uint8)

# Install skimage if needed
# !pip install scikit-image

# Test on one pair
common_id = enhanced_sample['GalaxyID'].iloc[0]
orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

matched = histogram_match_to_original(enh_path, orig_path)

fig, axes = plt.subplots(1, 3, figsize=(15,5))
axes[0].imshow(Image.open(orig_path))
axes[0].set_title('Original')
axes[1].imshow(Image.open(enh_path))
axes[1].set_title('Enhanced (raw)')
axes[2].imshow(matched)
axes[2].set_title('Enhanced (histogram matched)')
plt.show()

In [ ]:
common_id = enhanced_sample['GalaxyID'].iloc[0]
orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_id = os.path.basename(orig_path).replace('.jpg','')
enh_id  = re.search(r'(\d+)', os.path.basename(enh_path)).group(1)

print(f"Common ID selected:    {common_id}")
print(f"Original filename ID:  {orig_id}")
print(f"Enhanced filename ID:  {enh_id}")
print(f"Match: {'✅' if str(common_id) == orig_id == enh_id else '❌ MISMATCH'}")

In [ ]:
print(matched_kaggle_cols)

In [ ]:
# Rebuild ground truth df from sample and matched columns
ground_truth_df = sample[matched_kaggle_cols].reset_index(drop=True)
print(ground_truth_df.shape)
print(ground_truth_df.head())

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
all_classes = sorted(set(true_val_full.unique()) |
                     set(orig_classes_full.unique()) |
                     set(enh_classes_full.unique()))

fig, axes = plt.subplots(1, 2, figsize=(22, 9))

cm_orig = confusion_matrix(true_val_full, orig_classes_full, labels=all_classes)
disp_orig = ConfusionMatrixDisplay(cm_orig, display_labels=all_classes)
disp_orig.plot(ax=axes[0], colorbar=False, cmap='Blues', xticks_rotation=90)
axes[0].set_title(f'Original Images\nAccuracy: {acc_orig*100:.2f}%  RMSE: 0.10781')

cm_enh = confusion_matrix(true_val_full, enh_classes_full, labels=all_classes)
disp_enh = ConfusionMatrixDisplay(cm_enh, display_labels=all_classes)
disp_enh.plot(ax=axes[1], colorbar=False, cmap='Greens', xticks_rotation=90)
axes[1].set_title(f'Enhanced Images\nAccuracy: {acc_enh*100:.2f}%  RMSE: 0.11963')

plt.suptitle('Galaxy Morphology Classification — Full Decision Tree\nOriginal vs Quantum-Enhanced Images',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/confusion_matrix_full.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Install grad-cam library
!pip install grad-cam

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import torch
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Get the CNN part of the model (encoder)
# The last conv layer is what we target
# target_layer = [model.encoder.model.stages[-1].blocks[-1].conv_dw]
target_layer = [model.encoder.stages[3].blocks[-1].conv_dw]

cam = GradCAM(model=model.encoder, target_layers=target_layer)

def get_gradcam(image_path, transform, cam):
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0)

    targets = [ClassifierOutputTarget(0)]

    grayscale_cam = cam(
        input_tensor=tensor,
        targets=targets
    )

    # Generate CAM
    grayscale_cam = cam(input_tensor=tensor)
    grayscale_cam = grayscale_cam[0]

    # Overlay on image
    img_resized = np.array(img.resize((224,224))) / 255.0
    visualization = show_cam_on_image(
        img_resized.astype(np.float32),
        grayscale_cam,
        use_rgb=True
    )
    return grayscale_cam, visualization

# Compare original vs enhanced for same galaxy
common_id = enhanced_sample['GalaxyID'].iloc[0]
orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_cam, orig_vis = get_gradcam(orig_path, inference_transform, cam)
enh_cam, enh_vis   = get_gradcam(enh_path, inference_transform, cam)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(orig_vis)
axes[0].set_title('Grad-CAM — Original Image\n(where model looks)')
axes[1].imshow(enh_vis)
axes[1].set_title('Grad-CAM — Enhanced Image\n(where model looks)')
plt.suptitle('Model Attention Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
#plt.savefig('/content/mydrive/MyDrive/galaxy_images/gradcam_comparison.png',
            #dpi=150, bbox_inches='tight')
plt.show()

# Measure attention sharpness — how concentrated is the attention on galaxy center?
def attention_sharpness(cam_map):
    # Higher concentration near center = sharper attention
    h, w = cam_map.shape
    center_y, center_x = h//2, w//2
    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X-center_x)**2 + (Y-center_y)**2)
    # Weighted average distance of attention from center
    # Lower = more focused on center = better
    total_attention = cam_map.sum() + 1e-8
    weighted_dist = (cam_map * dist).sum() / total_attention
    return weighted_dist

orig_sharpness = attention_sharpness(orig_cam)
enh_sharpness  = attention_sharpness(enh_cam)

print(f"Attention center distance — Original:  {orig_sharpness:.4f}")
print(f"Attention center distance — Enhanced:  {enh_sharpness:.4f}")
print(f"Lower = more focused on galaxy center = better")
if enh_sharpness < orig_sharpness:
    print("✅ Enhanced images produce MORE focused model attention!")

In [ ]:
import cv2
import numpy as np

def edge_strength(image):

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Sobel gradients
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    magnitude = np.sqrt(gx**2 + gy**2)

    return magnitude.mean()

common_id = enhanced_sample['GalaxyID'].iloc[0]
orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_img = Image.open(orig_path).convert('RGB')
enh_img = Image.open(enh_path).convert('RGB')

print(f"Original image edge strength: {edge_strength(np.array(orig_img)):.4f}")
print(f"Enhanced image edge strength: {edge_strength(np.array(enh_img)):.4f}")


In [ ]:
import cv2
import numpy as np
from PIL import Image
from scipy import stats
import matplotlib.pyplot as plt

def edge_strength(image_path):
    img = np.array(Image.open(image_path).convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(gx**2 + gy**2)
    return magnitude.mean()

# Run on original sample
orig_edges = []
for path in sample['file_loc'].tolist():
    try:
        orig_edges.append(edge_strength(path))
    except:
        continue

# Run on enhanced sample
enh_edges = []
for path in enhanced_sample['file_loc'].tolist():
    try:
        enh_edges.append(edge_strength(path))
    except:
        continue

orig_edges = np.array(orig_edges)
enh_edges  = np.array(enh_edges)

print(f"=== EDGE STRENGTH ANALYSIS (n={len(orig_edges)} images) ===")
print(f"Original  — mean: {orig_edges.mean():.4f}, std: {orig_edges.std():.4f}")
print(f"Enhanced  — mean: {enh_edges.mean():.4f}, std: {enh_edges.std():.4f}")
print(f"Improvement ratio: {enh_edges.mean()/orig_edges.mean():.2f}x")

# Statistical significance
t_stat, p_value = stats.ttest_ind(orig_edges, enh_edges)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value:     {p_value:.6f}")
if p_value < 0.05:
    print("✅ Statistically significant (p < 0.05)")

# Plot
plt.figure(figsize=(10, 5))
plt.hist(orig_edges, bins=30, alpha=0.6,
         label=f'Original (mean={orig_edges.mean():.2f})', color='blue')
plt.hist(enh_edges,  bins=30, alpha=0.6,
         label=f'Enhanced (mean={enh_edges.mean():.2f})', color='green')
plt.xlabel('Edge Strength')
plt.ylabel('Number of Images')
plt.title('Edge Strength Distribution\nOriginal vs Quantum-Enhanced Galaxy Images')
plt.legend()
plt.show()

In [ ]:
def laplacian_variance(image):

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    lap = cv2.Laplacian(gray, cv2.CV_64F)

    return lap.var()

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_img = Image.open(orig_path).convert('RGB')
enh_img = Image.open(enh_path).convert('RGB')

print(f"Original image laplacian variance: {laplacian_variance(np.array(orig_img)):.4f}")
print(f"Enhanced image laplacian variance: {laplacian_variance(np.array(enh_img)):.4f}")

In [ ]:
def laplacian_variance(image_path):
    img = np.array(Image.open(image_path).convert('RGB'))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return lap.var()

# Run on original sample
orig_lap = []
for path in sample['file_loc'].tolist():
    try:
        orig_lap.append(laplacian_variance(path))
    except:
        continue

# Run on enhanced sample
enh_lap = []
for path in enhanced_sample['file_loc'].tolist():
    try:
        enh_lap.append(laplacian_variance(path))
    except:
        continue

orig_lap = np.array(orig_lap)
enh_lap  = np.array(enh_lap)

print(f"=== LAPLACIAN VARIANCE ANALYSIS (n={len(orig_lap)} images) ===")
print(f"Original  — mean: {orig_lap.mean():.4f}, std: {orig_lap.std():.4f}")
print(f"Enhanced  — mean: {enh_lap.mean():.4f}, std: {enh_lap.std():.4f}")
print(f"Improvement ratio: {enh_lap.mean()/orig_lap.mean():.2f}x")

# Statistical significance
t_stat, p_value = stats.ttest_ind(orig_lap, enh_lap)
print(f"\nT-statistic: {t_stat:.4f}")
print(f"P-value:     {p_value:.6f}")
if p_value < 0.05:
    print("✅ Statistically significant (p < 0.05)")

# Plot
plt.figure(figsize=(10, 5))
plt.hist(orig_lap, bins=30, alpha=0.6,
         label=f'Original (mean={orig_lap.mean():.2f})', color='blue')
plt.hist(enh_lap,  bins=30, alpha=0.6,
         label=f'Enhanced (mean={enh_lap.mean():.2f})', color='green')
plt.xlabel('Laplacian Variance')
plt.ylabel('Number of Images')
plt.title('Image Sharpness Distribution\nOriginal vs Quantum-Enhanced Galaxy Images')
plt.legend()
plt.show()

In [ ]:
import cv2
import numpy as np
from PIL import Image

def structure_coherence(image_path):

    # Load image
    img = Image.open(image_path).convert('RGB')
    img = np.array(img)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    gray = gray.astype(np.float32) / 255.0


    Ix = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    # Structure tensor components

    Jxx = cv2.GaussianBlur(Ix * Ix, (5,5), 1)
    Jyy = cv2.GaussianBlur(Iy * Iy, (5,5), 1)
    Jxy = cv2.GaussianBlur(Ix * Iy, (5,5), 1)

    # Eigenvalue computation
    trace = Jxx + Jyy
    det = Jxx * Jyy - Jxy**2

    temp = np.sqrt(np.maximum(trace**2 - 4*det, 0))

    lambda1 = (trace + temp) / 2
    lambda2 = (trace - temp) / 2


    coherence = ((lambda1 - lambda2)**2) / (
        (lambda1 + lambda2)**2 + 1e-8
    )

    # Mean coherence
    return coherence.mean()

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_score = structure_coherence(orig_path)
enh_score  = structure_coherence(enh_path)

print(f"Original structure coherence: {orig_score:.4f}")
print(f"Enhanced structure coherence: {enh_score:.4f}")

print("Higher = more organized directional morphology")

if enh_score > orig_score:
    print("✅ Enhanced image shows MORE coherent galaxy structure!")
else:
    print("❌ Original image shows more coherent structure")

In [ ]:
import cv2
import numpy as np
from PIL import Image

def galaxy_edge_snr(image_path):
    img = Image.open(image_path).convert('RGB')
    img = np.array(img)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Otsu threshold automatically finds galaxy region
    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Optional: smooth mask
    mask = cv2.GaussianBlur(mask, (9,9), 0)

    # Convert mask to boolean
    galaxy_mask = mask > 20

    # Background mask
    background_mask = ~galaxy_mask

    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    edge_mag = np.sqrt(gx**2 + gy**2)


    galaxy_edges = edge_mag[galaxy_mask]
    background_edges = edge_mag[background_mask]

    # Avoid divide-by-zero
    bg_mean = background_edges.mean() + 1e-8

    edge_snr = galaxy_edges.mean() / bg_mean

    return edge_snr

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_snr = galaxy_edge_snr(orig_path)
enh_snr  = galaxy_edge_snr(enh_path)

print(f"Original Edge-SNR:  {orig_snr:.4f}")
print(f"Enhanced Edge-SNR: {enh_snr:.4f}")

print("Higher = more morphology-focused enhancement")

if enh_snr > orig_snr:
    print("✅ Enhanced image strengthens galaxy structure more than background noise!")
else:
    print("❌ Enhancement amplifies background/noise relatively more")

orig_scores = []
enh_scores = []

for i in range(100):

    orig_path_j = sample.iloc[i]['file_loc']
    enh_path_j  = enhanced_sample.iloc[i]['file_loc']

    orig_scores.append(
        galaxy_edge_snr(orig_path_j)
    )

    enh_scores.append(
        galaxy_edge_snr(enh_path_j)
    )

print("Average Original Edge-SNR:",
      np.mean(orig_scores))

print("Average Enhanced Edge-SNR:",
      np.mean(enh_scores))

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


def localized_enhancement(
    orig_path,
    enh_path
):

    orig = Image.open(orig_path).convert('RGB')
    enh  = Image.open(enh_path).convert('RGB')

    plt.imshow(orig)
   # plt.imshow(enh)


    orig = np.array(orig)
    enh  = np.array(enh)

    # Resize enhanced image to original size
    enh = cv2.resize(
    enh,
    (orig.shape[1], orig.shape[0])
)


    gray = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)

    # Otsu threshold
    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Smooth mask edges
    mask = cv2.GaussianBlur(mask, (31,31), 0)

    # Normalize mask
    mask = mask.astype(np.float32) / 255.0

    # Expand to 3 channels
    mask = np.stack([mask]*3, axis=-1)

    result = (
        enh * mask +
        orig * (1-mask)
    )

    result = np.clip(result, 0, 255).astype(np.uint8)

    return result

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

result = localized_enhancement(
    orig_path,
    enh_path
)


In [ ]:
import cv2
import numpy as np
from PIL import Image

def localized_enhancement(orig_path, enh_path):

    orig = Image.open(orig_path).convert('RGB')
    enh  = Image.open(enh_path).convert('RGB')

    orig = np.array(orig)
    enh  = np.array(enh)

    enh = cv2.resize(
        enh,
        (orig.shape[1], orig.shape[0])
    )

    gray = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)

    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    mask = cv2.GaussianBlur(mask, (31,31), 0)

    mask = mask.astype(np.float32) / 255.0

    mask = np.stack([mask]*3, axis=-1)

    result = (
        enh * mask +
        orig * (1 - mask)
    )

    result = np.clip(result, 0, 255).astype(np.uint8)

    return result


def galaxy_edge_snr(img):

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    #galaxy mask create
    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    mask = cv2.GaussianBlur(mask, (9,9), 0)

    galaxy_mask = mask > 20
    background_mask = ~galaxy_mask


    # Edge magnitude
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    edge_mag = np.sqrt(gx**2 + gy**2)

    galaxy_edges = edge_mag[galaxy_mask]
    background_edges = edge_mag[background_mask]

    bg_mean = background_edges.mean() + 1e-8

    edge_snr = galaxy_edges.mean() / bg_mean

    return edge_snr


localized_img = localized_enhancement(
    orig_path,
    enh_path
)

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

# Load original image
orig_img = np.array(
    Image.open(orig_path).convert('RGB')
)

# Resize enhanced image to original size
enh_img = np.array(
    Image.open(enh_path).convert('RGB')
)

enh_img = cv2.resize(
    enh_img,
    (orig_img.shape[1], orig_img.shape[0])
)

# Compute SNRs
orig_snr = galaxy_edge_snr(orig_img)
enh_snr  = galaxy_edge_snr(enh_img)
local_snr = galaxy_edge_snr(localized_img)

print(f"Original Edge-SNR:   {orig_snr:.4f}")
print(f"Enhanced Edge-SNR:  {enh_snr:.4f}")
print(f"Localized Edge-SNR: {local_snr:.4f}")

print("\nHigher = more morphology-focused enhancement\n")

# Best result check
best = max(orig_snr, enh_snr, local_snr)

if best == local_snr:
    print("✅ Localized enhancement gives BEST morphology specificity!")

elif best == enh_snr:
    print("✅ Full enhancement gives BEST morphology specificity!")

else:
    print("✅ Original image remains most morphology-specific")


orig_scores = []
enh_scores = []
local_scores = []

N = 100   # number of samples

for i in range(N):

    orig_path = sample.iloc[i]['file_loc']
    enh_path  = enhanced_sample.iloc[i]['file_loc']

    # Load original
    orig_img = np.array(
        Image.open(orig_path).convert('RGB')
    )

    # Load enhanced
    enh_img = np.array(
        Image.open(enh_path).convert('RGB')
    )

    enh_img = cv2.resize(
        enh_img,
        (orig_img.shape[1], orig_img.shape[0])
    )

    # Localized blended image
    local_img = localized_enhancement(
        orig_path,
        enh_path
    )

    # Compute metrics
    orig_scores.append(
        galaxy_edge_snr(orig_img)
    )

    enh_scores.append(
        galaxy_edge_snr(enh_img)
    )

    local_scores.append(
        galaxy_edge_snr(local_img)
    )

print("\n========== DATASET RESULTS ==========\n")

print("Average Original Edge-SNR:  ",
      np.mean(orig_scores))

print("Average Enhanced Edge-SNR: ",
      np.mean(enh_scores))

print("Average Localized Edge-SNR:",
      np.mean(local_scores))

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy import stats

def localized_enhancement(orig_path, enh_path):
    orig = np.array(Image.open(orig_path).convert('RGB'))
    enh  = np.array(Image.open(enh_path).convert('RGB'))
    enh  = cv2.resize(enh, (orig.shape[1], orig.shape[0]))
    gray = cv2.cvtColor(orig, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.GaussianBlur(mask, (31,31), 0)
    mask = mask.astype(np.float32) / 255.0
    mask = np.stack([mask]*3, axis=-1)
    result = enh * mask + orig * (1 - mask)
    return np.clip(result, 0, 255).astype(np.uint8)

def galaxy_edge_snr(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.GaussianBlur(mask, (9,9), 0)
    galaxy_mask = mask > 20
    background_mask = ~galaxy_mask
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_mag = np.sqrt(gx**2 + gy**2)
    galaxy_edges = edge_mag[galaxy_mask]
    background_edges = edge_mag[background_mask]
    bg_mean = background_edges.mean() + 1e-8
    return galaxy_edges.mean() / bg_mean


orig_scores  = []
enh_scores   = []
local_scores = []
N = 100

for i in range(N):
    # orig_path_i = sample.iloc[i]['file_loc']
    # enh_path_i  = enhanced_sample.iloc[i]['file_loc']
    common_id_i = enhanced_sample['GalaxyID'].iloc[i]

    orig_path_i = sample[sample['GalaxyID'] == common_id_i]['file_loc'].iloc[0]

    enh_path_i = enhanced_sample[enhanced_sample['GalaxyID'] == common_id_i]['file_loc'].iloc[0]
    orig_img = np.array(Image.open(orig_path_i).convert('RGB'))
    enh_img  = cv2.resize(np.array(Image.open(enh_path_i).convert('RGB')),
                          (orig_img.shape[1], orig_img.shape[0]))
    local_img = localized_enhancement(orig_path_i, enh_path_i)
    orig_scores.append(galaxy_edge_snr(orig_img))
    enh_scores.append(galaxy_edge_snr(enh_img))
    local_scores.append(galaxy_edge_snr(local_img))

orig_scores  = np.array(orig_scores)
enh_scores   = np.array(enh_scores)
local_scores = np.array(local_scores)

# Statistical significance
_, p_orig_enh   = stats.ttest_rel(orig_scores, enh_scores)
_, p_orig_local = stats.ttest_rel(orig_scores, local_scores)
_, p_enh_local  = stats.ttest_rel(enh_scores,  local_scores)

print("\n========== DATASET RESULTS ==========")
print(f"Average Original Edge-SNR:   {orig_scores.mean():.4f} ± {orig_scores.std():.4f}")
print(f"Average Enhanced Edge-SNR:   {enh_scores.mean():.4f} ± {enh_scores.std():.4f}")
print(f"Average Localized Edge-SNR:  {local_scores.mean():.4f} ± {local_scores.std():.4f}")
print(f"\nOrig vs Enhanced p-value:    {p_orig_enh:.6f} {'✅' if p_orig_enh < 0.05 else '❌'}")
print(f"Orig vs Localized p-value:   {p_orig_local:.6f} {'✅' if p_orig_local < 0.05 else '❌'}")
print(f"Enhanced vs Localized p-val: {p_enh_local:.6f} {'✅' if p_enh_local < 0.05 else '❌'}")

# Plot 1 — Distribution histogram
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, scores, label, color in zip(
    axes,
    [orig_scores, enh_scores, local_scores],
    ['Original', 'Enhanced (Full)', 'Enhanced (Localized)'],
    ['steelblue', 'tomato', 'seagreen']
):
    ax.hist(scores, bins=20, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(scores.mean(), color='black', linestyle='--', linewidth=2,
               label=f'Mean: {scores.mean():.2f}')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Edge-SNR')
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Edge-SNR Distribution — Original vs Quantum-Enhanced Galaxy Images\n(n=100 galaxy pairs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot 2 — Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 6))

means  = [orig_scores.mean(), enh_scores.mean(), local_scores.mean()]
stds   = [orig_scores.std(),  enh_scores.std(),  local_scores.std()]
labels = ['Original', 'Enhanced\n(Full)', 'Enhanced\n(Localized)']
colors = ['steelblue', 'tomato', 'seagreen']

bars = ax.bar(labels, means, yerr=stds, capsize=8,
              color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)

# Annotate bars with mean values
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1,
            f'{mean:.2f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Average Edge-SNR', fontsize=12)
ax.set_title('Galaxy Edge-SNR Comparison\nOriginal vs Quantum-Enhanced Images',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(means) + max(stds) + 1)
ax.axhline(orig_scores.mean(), color='steelblue', linestyle=':', alpha=0.5)
plt.tight_layout()

plt.show()

# Plot 3 — Per-galaxy scatter comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original vs Enhanced
axes[0].scatter(orig_scores, enh_scores, alpha=0.5, color='tomato', s=30)
axes[0].plot([0, max(orig_scores.max(), enh_scores.max())],
             [0, max(orig_scores.max(), enh_scores.max())],
             'k--', linewidth=1, label='y=x (no change)')
axes[0].set_xlabel('Original Edge-SNR')
axes[0].set_ylabel('Enhanced Edge-SNR')
axes[0].set_title('Original vs Full Enhanced\n(points above line = enhanced wins)')
axes[0].legend()

# Original vs Localized
axes[1].scatter(orig_scores, local_scores, alpha=0.5, color='seagreen', s=30)
axes[1].plot([0, max(orig_scores.max(), local_scores.max())],
             [0, max(orig_scores.max(), local_scores.max())],
             'k--', linewidth=1, label='y=x (no change)')
axes[1].set_xlabel('Original Edge-SNR')
axes[1].set_ylabel('Localized Edge-SNR')
axes[1].set_title('Original vs Localized Enhanced\n(points above line = localized wins)')
axes[1].legend()

plt.suptitle('Per-Galaxy Edge-SNR Comparison (n=100)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot 4 — Visual example (3 images side by side)
common_id = enhanced_sample['GalaxyID'].iloc[0]
example_orig  = np.array(Image.open(sample[sample['GalaxyID']==common_id]['file_loc'].values[0]).convert('RGB'))
example_enh   = cv2.resize(
    np.array(Image.open(enhanced_sample.iloc[0]['file_loc']).convert('RGB')),
    (example_orig.shape[1], example_orig.shape[0])
)


example_local = localized_enhancement(
    sample[sample['GalaxyID']==common_id]['file_loc'].values[0],
    enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]
)

snr_o = galaxy_edge_snr(example_orig)
snr_e = galaxy_edge_snr(example_enh)
snr_l = galaxy_edge_snr(example_local)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title, snr in zip(
    axes,
    [example_orig, example_enh, example_local],
    ['Original', 'Enhanced (Full)', 'Enhanced (Localized)'],
    [snr_o, snr_e, snr_l]
):
    ax.imshow(img)
    ax.set_title(f'{title}\nEdge-SNR: {snr:.4f}', fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Visual Comparison — Galaxy Image Enhancement\n(GalaxyID: {})'.format(
    sample.iloc[0]['GalaxyID']), fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
from PIL import Image

def structure_coherence(img):

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    gray = gray.astype(np.float32) / 255.0

    # Compute image gradients
    Ix = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

    # Structure tensor components
    Jxx = cv2.GaussianBlur(Ix * Ix, (5,5), 1)
    Jyy = cv2.GaussianBlur(Iy * Iy, (5,5), 1)
    Jxy = cv2.GaussianBlur(Ix * Iy, (5,5), 1)

    # Eigen value computation
    trace = Jxx + Jyy
    det = Jxx * Jyy - Jxy**2

    temp = np.sqrt(
        np.maximum(trace**2 - 4*det, 0)
    )

    lambda1 = (trace + temp) / 2
    lambda2 = (trace - temp) / 2


    coherence = (
        ((lambda1 - lambda2)**2) /
        ((lambda1 + lambda2)**2 + 1e-8)
    )

    return coherence.mean()

orig_path = sample[sample['GalaxyID']==common_id]['file_loc'].values[0]
enh_path  = enhanced_sample[enhanced_sample['GalaxyID']==common_id]['file_loc'].values[0]

orig_img = np.array(
    Image.open(orig_path).convert('RGB')
)

enh_img = np.array(
    Image.open(enh_path).convert('RGB')
)

# Resize enhanced to original size
enh_img = cv2.resize(
    enh_img,
    (orig_img.shape[1], orig_img.shape[0])
)


localized_img = localized_enhancement(
    orig_path,
    enh_path
)


orig_score = structure_coherence(orig_img)

enh_score = structure_coherence(enh_img)

local_score = structure_coherence(localized_img)


print(f"Original structure coherence:   {orig_score:.4f}")

print(f"Enhanced structure coherence:  {enh_score:.4f}")

print(f"Localized structure coherence: {local_score:.4f}")

print("\nHigher = more organized directional morphology\n")

best = max(orig_score, enh_score, local_score)

if best == local_score:
    print("✅ Localized enhancement preserves structure BEST!")

elif best == enh_score:
    print("✅ Full enhancement preserves structure BEST!")

else:
    print("✅ Original image remains most structurally coherent")

In [ ]:
import cv2
import numpy as np
from PIL import Image
from scipy import stats
import matplotlib.pyplot as plt

def structure_coherence(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    Ix = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    Jxx = cv2.GaussianBlur(Ix * Ix, (5,5), 1)
    Jyy = cv2.GaussianBlur(Iy * Iy, (5,5), 1)
    Jxy = cv2.GaussianBlur(Ix * Iy, (5,5), 1)
    trace = Jxx + Jyy
    det = Jxx * Jyy - Jxy**2
    temp = np.sqrt(np.maximum(trace**2 - 4*det, 0))
    lambda1 = (trace + temp) / 2
    lambda2 = (trace - temp) / 2
    coherence = ((lambda1 - lambda2)**2) / ((lambda1 + lambda2)**2 + 1e-8)
    return coherence.mean()

def edge_strength(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return np.sqrt(gx**2 + gy**2).mean()

def laplacian_variance(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def load_image(path, target_shape=None):
    img = np.array(Image.open(path).convert('RGB'))
    if target_shape is not None and img.shape != target_shape:
        img = cv2.resize(img, (target_shape[1], target_shape[0]))
    return img

# Run across all matched pairs
orig_edge, enh_edge = [], []
orig_lap,  enh_lap  = [], []
orig_coh,  enh_coh  = [], []

# Only use GalaxyIDs present in both samples
common_ids = set(sample['GalaxyID']) & set(enhanced_sample['GalaxyID'])
print(f"Common galaxy pairs: {len(common_ids)}")

for gid in list(common_ids):
    try:
        op = sample[sample['GalaxyID']==gid]['file_loc'].values[0]
        ep = enhanced_sample[enhanced_sample['GalaxyID']==gid]['file_loc'].values[0]

        orig_arr = load_image(op)
        enh_arr  = load_image(ep, target_shape=orig_arr.shape)

        orig_edge.append(edge_strength(orig_arr))
        enh_edge.append(edge_strength(enh_arr))

        orig_lap.append(laplacian_variance(orig_arr))
        enh_lap.append(laplacian_variance(enh_arr))

        orig_coh.append(structure_coherence(orig_arr))
        enh_coh.append(structure_coherence(enh_arr))

    except Exception as e:
        continue

# Convert to arrays
orig_edge = np.array(orig_edge)
enh_edge  = np.array(enh_edge)
orig_lap  = np.array(orig_lap)
enh_lap   = np.array(enh_lap)
orig_coh  = np.array(orig_coh)
enh_coh   = np.array(enh_coh)

# Print results
print("\n=== IMAGE QUALITY METRICS (full sample) ===\n")

for name, o, e in [
    ('Edge Strength',       orig_edge, enh_edge),
    ('Laplacian Variance',  orig_lap,  enh_lap),
    ('Structure Coherence', orig_coh,  enh_coh),
]:
    t_stat, p_val = stats.ttest_ind(o, e)
    ratio = e.mean() / o.mean()
    winner = "Enhanced ✅" if e.mean() > o.mean() else "Original ✅"
    print(f"{name}")
    print(f"  Original: {o.mean():.4f} ± {o.std():.4f}")
    print(f"  Enhanced: {e.mean():.4f} ± {e.std():.4f}")
    print(f"  Ratio:    {ratio:.2f}x")
    print(f"  P-value:  {p_val:.6f} {'✅ significant' if p_val < 0.05 else '❌ not significant'}")
    print(f"  Winner:   {winner}\n")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, o, e) in enumerate([
    ('Edge Strength',       orig_edge, enh_edge),
    ('Laplacian Variance',  orig_lap,  enh_lap),
    ('Structure Coherence', orig_coh,  enh_coh),
]):
    axes[idx].hist(o, bins=30, alpha=0.6,
                   label=f'Original\n(mean={o.mean():.2f})', color='blue')
    axes[idx].hist(e, bins=30, alpha=0.6,
                   label=f'Enhanced\n(mean={e.mean():.2f})', color='green')
    axes[idx].set_xlabel(name)
    axes[idx].set_ylabel('Count')
    axes[idx].legend(fontsize=8)
    axes[idx].set_title(name)

plt.suptitle('Image Quality Metrics — Original vs Quantum-Enhanced\n(n={} galaxy pairs)'.format(len(common_ids)),
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class LocalizedGalaxyDataset(Dataset):

    def __init__(
        self,
        orig_df,
        enh_df,
        transform=None
    ):

        self.orig_df = orig_df.reset_index(drop=True)
        self.enh_df  = enh_df.reset_index(drop=True)

        self.transform = transform

    def __len__(self):
        return len(self.orig_df)

    def __getitem__(self, idx):

        orig_path = self.orig_df.iloc[idx]['file_loc']
        enh_path  = self.enh_df.iloc[idx]['file_loc']

        localized_img = localized_enhancement(
            orig_path,
            enh_path
        )

        img = Image.fromarray(localized_img)

        if self.transform:
            img = self.transform(img)

        return img

localized_dataset = LocalizedGalaxyDataset(
    sample,
    enhanced_sample,
    transform=enhanced_inference_transform
)

localized_loader = DataLoader(
    localized_dataset,
    batch_size=32,
    shuffle=False
)

print(f"Localized dataset ready: {len(localized_dataset)} images")

localized_features_list = []

encoder.eval()

with torch.no_grad():

    for batch in localized_loader:

        # batch = batch.to(device)

        feats = encoder(batch)

        localized_features_list.append(
            feats.cpu().numpy()
        )

localized_features = np.concatenate(
    localized_features_list,
    axis=0
)

print(f"Localized features shape: {localized_features.shape}")

from sklearn.model_selection import train_test_split

labels = sample.drop(
    columns=['GalaxyID', 'file_loc']
).values

X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    localized_features,
    labels,
    test_size=0.2,
    random_state=42
)

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

regressor_local_5 = MultiOutputRegressor(
    Ridge(alpha=5.0)
)

regressor_local_5.fit(
    X_train_local,
    y_train_local
)

y_pred_local_5 = np.clip(
    regressor_local_5.predict(X_val_local),
    0,
    1
)

rmse_local_5 = np.sqrt(
    np.mean(
        (y_pred_local_5 - y_val_local) ** 2
    )
)

print(f"\nAlpha=5.0 Results:")

print(f"Original images RMSE:              {rmse_orig_5:.5f}")

print(f"Enhanced cross-domain RMSE:        0.28318")

print(f"Enhanced fine-tuned RMSE:          {rmse_enh_5:.5f}")

print(f"Localized enhanced RMSE:           {rmse_local_5:.5f}")

print(f"Dieleman winner:                   0.07492")

print(f"Central pixel baseline:            0.16194")

In [ ]:
# Install grad-cam library
!pip install grad-cam

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layer = [
    model.encoder.stages[3].blocks[-1].conv_dw
]

cam = GradCAM(
    model=model.encoder,
    target_layers=target_layer
)

def get_gradcam_from_array(img_array, transform, cam):

    pil_img = Image.fromarray(img_array)

    tensor = transform(pil_img).unsqueeze(0)

    tensor = tensor.to(device)

    targets = [ClassifierOutputTarget(0)]

    grayscale_cam = cam(
        input_tensor=tensor,
        targets=targets
    )

    grayscale_cam = grayscale_cam[0]

    img_resized = np.array(
        pil_img.resize((224,224))
    ) / 255.0

    visualization = show_cam_on_image(
        img_resized.astype(np.float32),
        grayscale_cam,
        use_rgb=True
    )

    return grayscale_cam, visualization

common_id = enhanced_sample['GalaxyID'].iloc[0]

orig_path = sample[
    sample['GalaxyID']==common_id
]['file_loc'].values[0]

enh_path = enhanced_sample[
    enhanced_sample['GalaxyID']==common_id
]['file_loc'].values[0]

orig_img = np.array(
    Image.open(orig_path).convert('RGB')
)

enh_img = np.array(
    Image.open(enh_path).convert('RGB')
)

enh_img = cv2.resize(
    enh_img,
    (orig_img.shape[1], orig_img.shape[0])
)

localized_img = localized_enhancement(
    orig_path,
    enh_path
)

orig_cam, orig_vis = get_gradcam_from_array(
    orig_img,
    inference_transform,
    cam
)

enh_cam, enh_vis = get_gradcam_from_array(
    enh_img,
    enhanced_inference_transform,
    cam
)

local_cam, local_vis = get_gradcam_from_array(
    localized_img,
    enhanced_inference_transform,
    cam
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(orig_vis)
axes[0].set_title(
    'Original Image\nModel Attention'
)

axes[1].imshow(enh_vis)
axes[1].set_title(
    'Full Enhanced Image\nModel Attention'
)

axes[2].imshow(local_vis)
axes[2].set_title(
    'Localized Enhanced Image\nModel Attention'
)

for ax in axes:
    ax.axis('off')

plt.suptitle(
    'Grad-CAM Attention Comparison',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

def attention_sharpness(cam_map):

    h, w = cam_map.shape

    center_y, center_x = h//2, w//2

    Y, X = np.ogrid[:h, :w]

    dist = np.sqrt(
        (X-center_x)**2 +
        (Y-center_y)**2
    )

    # Weighted average attention distance
    total_attention = cam_map.sum() + 1e-8

    weighted_dist = (
        (cam_map * dist).sum() /
        total_attention
    )

    return weighted_dist


orig_sharpness = attention_sharpness(orig_cam)

enh_sharpness = attention_sharpness(enh_cam)

local_sharpness = attention_sharpness(local_cam)

print(f"Attention center distance — Original:   {orig_sharpness:.4f}")

print(f"Attention center distance — Enhanced:  {enh_sharpness:.4f}")

print(f"Attention center distance — Localized: {local_sharpness:.4f}")

print("\nLower = more focused on galaxy center\n")

best = min(
    orig_sharpness,
    enh_sharpness,
    local_sharpness
)

if best == local_sharpness:
    print("✅ Localized enhancement gives BEST attention focus!")

elif best == enh_sharpness:
    print("✅ Full enhancement gives BEST attention focus!")

else:
    print("✅ Original image gives BEST attention focus!")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt
import numpy as np

n_clusters = 5

# ORIGINAL FEATURES CLUSTERING
kmeans_orig = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

orig_clusters = kmeans_orig.fit_predict(
    features
)

# Silhouette score
sil_orig = silhouette_score(
    features,
    orig_clusters
)

print(f"Original feature silhouette score: {sil_orig:.4f}")

In [ ]:
# ENHANCED FEATURES CLUSTERING
kmeans_enh = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

enh_clusters = kmeans_enh.fit_predict(
    enhanced_features
)

# Silhouette score
sil_enh = silhouette_score(
    enhanced_features,
    enh_clusters
)

print(f"Enhanced feature silhouette score: {sil_enh:.4f}")

In [ ]:
# ENHANCED FEATURES CLUSTERING

kmeans_enh_loc = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

enh_clusters_loc = kmeans_enh_loc.fit_predict(
    localized_features
)

# Silhouette score
sil_enh_loc = silhouette_score(
    localized_features,
    enh_clusters_loc
)

print(f"Enhanced feature silhouette score: {sil_enh_loc:.4f}")

In [ ]:
pca = PCA(n_components=2)

orig_2d = pca.fit_transform(
    features
)

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    orig_2d[:,0],
    orig_2d[:,1],
    c=orig_clusters,
    s=20
)

plt.title("Original Galaxy Feature Clusters")
plt.xlabel("PCA-1")
plt.ylabel("PCA-2")

plt.colorbar(scatter)

plt.show()

In [ ]:
enh_2d = pca.fit_transform(
    enhanced_features
)

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    enh_2d[:,0],
    enh_2d[:,1],
    c=enh_clusters,
    s=20
)

plt.title("Enhanced Galaxy Feature Clusters")
plt.xlabel("PCA-1")
plt.ylabel("PCA-2")

plt.colorbar(scatter)

plt.show()

In [ ]:
enh_2d_loc = pca.fit_transform(
    localized_features
)

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    enh_2d_loc[:,0],
    enh_2d_loc[:,1],
    c=enh_clusters_loc,
    s=20
)

plt.title("Enhanced Galaxy Feature Clusters")
plt.xlabel("PCA-1")
plt.ylabel("PCA-2")

plt.colorbar(scatter)

plt.show()

In [ ]:
import cv2
import numpy as np
from PIL import Image
from skimage.exposure import match_histograms

def enhancement_v2(
    orig_path,
    enh_path,

    # Blend strengths
    galaxy_enh_strength=0.30,
    background_strength=0.0,    # keep background original

    # Mask softness
    blur_kernel=51,

    # Optional mild smoothing
    post_smooth=True,
    smooth_sigma=0.6
):

    #Load Images

    orig = np.array(
        Image.open(orig_path).convert('RGB')
    )

    enh = np.array(
        Image.open(enh_path).convert('RGB')
    )

    #Match size

    enh = cv2.resize(
        enh,
        (orig.shape[1], orig.shape[0])
    )

    # HISTOGRAM MATCHING

    enh = match_histograms(
        enh,
        orig,
        channel_axis=-1
    )

    enh = np.clip(enh, 0, 255).astype(np.uint8)

    # CREATE GALAXY MASK

    gray = cv2.cvtColor(
        orig,
        cv2.COLOR_RGB2GRAY
    )

    # Otsu threshold
    _, mask = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    # Large blur for soft transition
    mask = cv2.GaussianBlur(
        mask,
        (blur_kernel, blur_kernel),
        0
    )

    # Normalize mask to [0,1]
    mask = mask.astype(np.float32) / 255.0

    # Expand to 3 channels
    mask = np.stack([mask]*3, axis=-1)

    # GALAXY-ONLY BLENDING

    # Controlled enhancement
    galaxy_blend = (
        (1 - galaxy_enh_strength) * orig +
        galaxy_enh_strength * enh
    )

    # Preserve original background
    background_blend = (
        (1 - background_strength) * orig +
        background_strength * enh
    )

    # Final blend using mask
    final = (
        mask * galaxy_blend +
        (1 - mask) * background_blend
    )
#Optional Light Smoothing

    if post_smooth:

        final = cv2.GaussianBlur(
            final,
            (3,3),
            smooth_sigma
        )

    final = np.clip(final, 0, 255).astype(np.uint8)

    return final

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class LocalizedGalaxyDataset(Dataset):

    def __init__(
        self,
        orig_df,
        enh_df,
        transform=None
    ):

        self.orig_df = orig_df.reset_index(drop=True)
        self.enh_df  = enh_df.reset_index(drop=True)

        self.transform = transform

    def __len__(self):
        return len(self.orig_df)

    def __getitem__(self, idx):

        orig_path = self.orig_df.iloc[idx]['file_loc']
        enh_path  = self.enh_df.iloc[idx]['file_loc']

        localized_img = enhancement_v2(
            orig_path,
            enh_path
        )

        img = Image.fromarray(localized_img)

        if self.transform:
            img = self.transform(img)

        return img

localized_dataset = LocalizedGalaxyDataset(
    sample,
    enhanced_sample,
    transform=enhanced_inference_transform
)

localized_loader = DataLoader(
    localized_dataset,
    batch_size=32,
    shuffle=False
)

print(f"Localized dataset ready: {len(localized_dataset)} images")

localized_features_list = []

encoder.eval()

with torch.no_grad():

    for batch in localized_loader:

        # batch = batch.to(device)

        feats = encoder(batch)

        localized_features_list.append(
            feats.cpu().numpy()
        )

localized_features = np.concatenate(
    localized_features_list,
    axis=0
)

print(f"Localized features shape: {localized_features.shape}")

from sklearn.model_selection import train_test_split

labels = sample.drop(
    columns=['GalaxyID', 'file_loc']
).values

X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
    localized_features,
    labels,
    test_size=0.2,
    random_state=42
)

from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor

regressor_local_5 = MultiOutputRegressor(
    Ridge(alpha=5.0)
)

regressor_local_5.fit(
    X_train_local,
    y_train_local
)

y_pred_local_5 = np.clip(
    regressor_local_5.predict(X_val_local),
    0,
    1
)

rmse_enh_v2_5 = np.sqrt(
    np.mean(
        (y_pred_local_5 - y_val_local) ** 2
    )
)

print(f"\nAlpha=5.0 Results:")

print(f"Original images RMSE:              {rmse_orig_5:.5f}")

print(f"Enhanced cross-domain RMSE:        0.28318")

print(f"Enhanced fine-tuned RMSE:          {rmse_enh_5:.5f}")

print(f"Localized enhanced RMSE:           {rmse_local_5:.5f}")


print(f"Weighted enhancement images RMSE:     {rmse_enh_v2_5:.5f}")

print(f"Dieleman winner:                   0.07492")

print(f"Central pixel baseline:            0.16194")

In [ ]:
def assign_morphology_from_votes(row):

    smooth_vote = row['Class1.1']
    featured_vote = row['Class1.2']
    artifact_vote = row['Class1.3']

    edgeon_yes = row['Class2.1']
    edgeon_no  = row['Class2.2']

    spiral_yes = row['Class4.1']
    spiral_no  = row['Class4.2']

    merger_vote = row['Class8.3']

    # MERGER
    if merger_vote >= 0.50:
        return "Merger"

    # SPIRAL
    elif (
        featured_vote >= 0.45 and
        spiral_yes >= 0.50
    ):
        return "Spiral"

    # LENTICULAR / EDGE-ON DISK
    elif (
        featured_vote >= 0.45 and
        edgeon_yes >= 0.60
    ):
        return "Lenticular"

    # ELLIPTICAL / SMOOTH
    elif smooth_vote >= 0.60:
        return "Elliptical"

    # OTHERWISE
    else:
        return "Irregular"

In [ ]:
sample['morphology_label'] = sample.apply(
    assign_morphology_from_votes,
    axis=1
)

enhanced_with_labels['morphology_label'] = enhanced_with_labels.apply(
    assign_morphology_from_votes,
    axis=1
)

print(
    sample['morphology_label'].value_counts()
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_labels = label_encoder.fit_transform(
    sample['morphology_label']
)

print("Classes:")
print(label_encoder.classes_)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_orig = LogisticRegression(
    max_iter=5000
)

clf_orig.fit(
    X_train_orig,
    y_train
)

y_pred_orig = clf_orig.predict(
    X_test_orig
)

acc_orig = accuracy_score(
    y_test,
    y_pred_orig
)

print("ORIGINAL IMAGE CLASSIFICATION")

print(f"Accuracy: {acc_orig:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=label_encoder.classes_
    )
)

In [ ]:
X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    enhanced_features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_enh = LogisticRegression(
    max_iter=5000
)

clf_enh.fit(
    X_train_enh,
    y_train_enh
)

y_pred_enh = clf_enh.predict(
    X_test_enh
)

acc_enh = accuracy_score(
    y_test_enh,
    y_pred_enh
)

print("ENHANCED IMAGE CLASSIFICATION")

print(f"Accuracy: {acc_enh:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_enh,
        y_pred_enh,
        target_names=label_encoder.classes_
    )
)

In [ ]:
X_train_loc, X_test_loc, y_train_loc, y_test_loc = train_test_split(
    localized_features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_loc = LogisticRegression(
    max_iter=5000
)

clf_loc.fit(
    X_train_loc,
    y_train_loc
)

y_pred_loc = clf_loc.predict(
    X_test_loc
)

acc_loc = accuracy_score(
    y_test_loc,
    y_pred_loc
)

print("LOCALIZED ENHANCED IMAGE CLASSIFICATION")

print(f"Accuracy: {acc_loc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_loc,
        y_pred_loc,
        target_names=label_encoder.classes_
    )
)

In [ ]:
# We classify:
# barred
# edge-on
# ringed
# disturbed
# clumpy


def assign_auxiliary_attributes(row):

    attrs = []
    # BARRED
    # Class3.1 = bar yes


    if row['Class3.1'] >= 0.5:
        attrs.append("barred")


    # EDGE-ON
    # Class2.1 = edge-on yes


    if row['Class2.1'] >= 0.5:
        attrs.append("edge-on")


    # RINGED
    # Class7.1 = ring yes


    if row['Class7.1'] >= 0.5:
        attrs.append("ringed")


    # DISTURBED / MERGING
    # Class8.3 = merger


    if row['Class8.3'] >= 0.4:
        attrs.append("disturbed")


    # CLUMPY / IRREGULAR FEATURES
    # Approximation using odd features


    odd_total = (
        row['Class7.1'] +
        row['Class7.2'] +
        row['Class7.3']
    )

    if odd_total >= 0.8:
        attrs.append("clumpy")


    # If nothing assigned


    if len(attrs) == 0:
        attrs.append("normal")

    return attrs

In [ ]:
enhanced_with_labels['auxiliary_labels'] = (
    enhanced_with_labels.apply(
        assign_auxiliary_attributes,
        axis=1
    )
)

sample_with_labels = sample.merge(
    solutions,
    on='GalaxyID'
)

sample['auxiliary_labels'] = (
    sample.apply(
        assign_auxiliary_attributes,
        axis=1
    )
)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

y_aux = mlb.fit_transform(
    enhanced_with_labels['auxiliary_labels']
)

print("Auxiliary Classes:")
print(mlb.classes_)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    features,
    y_aux,
    test_size=0.2,
    random_state=42
)

clf_orig = OneVsRestClassifier(
    LogisticRegression(max_iter=5000)
)

clf_orig.fit(
    X_train_orig,
    y_train
)

y_pred_orig = clf_orig.predict(
    X_test_orig
)

f1_orig = f1_score(
    y_test,
    y_pred_orig,
    average='micro'
)


print("ORIGINAL AUXILIARY CLASSIFICATION")


print(f"Micro-F1 Score: {f1_orig:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=mlb.classes_
    )
)

In [ ]:
X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    enhanced_features,
    y_aux,
    test_size=0.2,
    random_state=42
)

clf_enh = OneVsRestClassifier(
    LogisticRegression(max_iter=5000)
)

clf_enh.fit(
    X_train_enh,
    y_train_enh
)


y_pred_enh = clf_enh.predict(
    X_test_enh
)

f1_enh = f1_score(
    y_test_enh,
    y_pred_enh,
    average='micro'
)


print("ENHANCED AUXILIARY CLASSIFICATION")


print(f"Micro-F1 Score: {f1_enh:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_enh,
        y_pred_enh,
        target_names=mlb.classes_
    )
)

In [ ]:

print("FINAL AUXILIARY COMPARISON")


print(f"Original Micro-F1:  {f1_orig:.4f}")
print(f"Enhanced Micro-F1: {f1_enh:.4f}")

if f1_enh > f1_orig:
    print("\n✅ Enhancement improves auxiliary morphology classification")
else:
    print("\n❌ Original images classify auxiliary attributes better")

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    f1_score
)

import matplotlib.pyplot as plt

acc_orig = accuracy_score(
    y_test,
    y_pred_orig
)

acc_enh = accuracy_score(
    y_test_enh,
    y_pred_enh
)

print("AUXILIARY ATTRIBUTE CLASSIFICATION")
print(f"Original Exact-Match Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Exact-Match Accuracy: {acc_enh:.4f}")

f1_orig = f1_score(
    y_test,
    y_pred_orig,
    average='micro'
)

f1_enh = f1_score(
    y_test_enh,
    y_pred_enh,
    average='micro'
)

print(f"\nOriginal Micro-F1 Score:  {f1_orig:.4f}")
print(f"Enhanced Micro-F1 Score: {f1_enh:.4f}")

print("ORIGINAL CLASSIFICATION REPORT")

print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=mlb.classes_
    )
)
print("ENHANCED CLASSIFICATION REPORT")

print(
    classification_report(
        y_test_enh,
        y_pred_enh,
        target_names=mlb.classes_
    )
)

for i, class_name in enumerate(mlb.classes_):

    cm_orig = confusion_matrix(
    y_test[:, i],
    y_pred_orig[:, i],
    labels=[0,1]
    )

    cm_enh = confusion_matrix(
    y_test_enh[:, i],
    y_pred_enh[:, i],
    labels=[0,1]
    )


    fig, axes = plt.subplots(1, 2, figsize=(10,4))


    disp1 = ConfusionMatrixDisplay(
        confusion_matrix=cm_orig,
        display_labels=["No", "Yes"]
    )

    disp1.plot(ax=axes[0])

    axes[0].set_title(
        f'Original — {class_name}'
    )


    disp2 = ConfusionMatrixDisplay(
        confusion_matrix=cm_enh,
        display_labels=["No", "Yes"]
    )

    disp2.plot(ax=axes[1])

    axes[1].set_title(
        f'Enhanced — {class_name}'
    )

    plt.tight_layout()
    plt.show()


print("FINAL COMPARISON")
print(f"Original Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Accuracy: {acc_enh:.4f}")

print(f"\nOriginal Micro-F1:  {f1_orig:.4f}")
print(f"Enhanced Micro-F1: {f1_enh:.4f}")

if f1_enh > f1_orig:
    print("\n✅ Enhanced images improve auxiliary classification")
else:
    print("\n❌ Original images classify auxiliary attributes better")

In [ ]:
def classify_galaxy(row):

    # Priority 1: Artifacts / Star
    if row["Class1.3"] > 0.5:
        return "Irregular"

    # Priority 2: Merger
    if row["Class11.1"] > 0.5:
        return "Irregular"

    # Priority 3: Edge-On
    # Class2.2 = Edge-on
    # Class2.1 = Not Edge-on
    if row["Class2.2"] > row["Class2.1"]:
        return "EdgeOn"

    # Priority 4: Morphology
    # Class1.1 = Smooth
    # Class1.2 = Disk/Features
    if row["Class1.1"] >= row["Class1.2"]:
        return "Elliptical"

    else:

        # Disk confirmed
        # Class4.1 = Spiral
        # Class4.2 = Lenticular

        if row["Class4.1"] > row["Class4.2"]:
            return "Spiral"

        else:
            return "Elliptical"


final_pred_df_orig["morphology_label"] = (
    final_pred_df_orig.apply(
        classify_galaxy,
        axis=1
    )
)

final_pred_df["morphology_label"] = (
    final_pred_df.apply(
        classify_galaxy,
        axis=1
    )
)

print("Original Dataset Class Distribution:")
print(
    final_pred_df_orig["morphology_label"].value_counts()
)

print("\nEnhanced Dataset Class Distribution:")
print(
    final_pred_df["morphology_label"].value_counts()
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

label_encoder = LabelEncoder()

y_labels = label_encoder.fit_transform(
    sample["morphology_label"]
)

print("Classes:")
print(label_encoder.classes_)


X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_orig = LogisticRegression(
    max_iter=5000
)

clf_orig.fit(
    X_train_orig,
    y_train
)

# Predictions
y_pred_orig = clf_orig.predict(
    X_test_orig
)

# Accuracy
acc_orig = accuracy_score(
    y_test,
    y_pred_orig
)

print("ORIGINAL CLASSIFICATION")
print(f"Accuracy: {acc_orig:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=label_encoder.classes_
    )
)
cm_orig = confusion_matrix(
    y_test,
    y_pred_orig
)

plt.figure(figsize=(7,6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_orig,
    display_labels=label_encoder.classes_
)

disp.plot(cmap='Blues')

plt.title("Original Image Classification")
plt.show()

X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    enhanced_features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_enh = LogisticRegression(
    max_iter=5000
)

clf_enh.fit(
    X_train_enh,
    y_train_enh
)

# Predictions
y_pred_enh = clf_enh.predict(
    X_test_enh
)

# Accuracy
acc_enh = accuracy_score(
    y_test_enh,
    y_pred_enh
)

print(f"Accuracy: {acc_enh:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_enh,
        y_pred_enh,
        target_names=label_encoder.classes_
    )
)

cm_enh = confusion_matrix(
    y_test_enh,
    y_pred_enh
)

plt.figure(figsize=(7,6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_enh,
    display_labels=label_encoder.classes_
)

disp.plot(cmap='Blues')

plt.title("Enhanced Image Classification")
plt.show()

print("FINAL COMPARISON")

print(f"Original Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:
    print("\n✅ Enhanced images classify galaxies better")
else:
    print("\n❌ Original images classify galaxies better")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


label_encoder = LabelEncoder()

y_labels = label_encoder.fit_transform(
    sample["morphology_label"]
)

print("Classes:")
print(label_encoder.classes_)


X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_orig = LogisticRegression(
    max_iter=5000
)

clf_orig.fit(
    X_train_orig,
    y_train
)

# Predictions
y_pred_orig = clf_orig.predict(
    X_test_orig
)

# Accuracy
acc_orig = accuracy_score(
    y_test,
    y_pred_orig
)


print(f"Accuracy: {acc_orig:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=label_encoder.classes_
    )
)

cm_orig = confusion_matrix(
    y_test,
    y_pred_orig
)

plt.figure(figsize=(7,6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_orig,
    display_labels=label_encoder.classes_
)

disp.plot(cmap='Blues')

plt.title("Original Image Classification")
plt.show()

X_train_enh, X_test_enh, y_train_enh, y_test_enh = train_test_split(
    enhanced_features,
    y_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

clf_enh = LogisticRegression(
    max_iter=5000
)

clf_enh.fit(
    X_train_enh,
    y_train_enh
)

# Predictions
y_pred_enh = clf_enh.predict(
    X_test_enh
)

# Accuracy
acc_enh = accuracy_score(
    y_test_enh,
    y_pred_enh
)

print("ENHANCED CLASSIFICATION")
print(f"Accuracy: {acc_enh:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_enh,
        y_pred_enh,
        target_names=label_encoder.classes_
    )
)

cm_enh = confusion_matrix(
    y_test_enh,
    y_pred_enh
)

plt.figure(figsize=(7,6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_enh,
    display_labels=label_encoder.classes_
)

disp.plot(cmap='Blues')

plt.title("Enhanced Image Classification")
plt.show()

print("FINAL COMPARISON")

print(f"Original Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:
    print("\n✅ Enhanced images classify galaxies better")
else:
    print("\n❌ Original images classify galaxies better")

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt
import numpy as np


label_encoder = LabelEncoder()

# Ground-truth labels from ORIGINAL dataframe
y_true = label_encoder.fit_transform(
    final_pred_df_orig["morphology_label"]
)

# Predicted labels from ENHANCED dataframe
y_pred = label_encoder.transform(
    final_pred_df["morphology_label"]
)

print("Classes:")
print(label_encoder.classes_)


accuracy = accuracy_score(
    y_true,
    y_pred
)


print("CLASSIFICATION RESULTS")


print(f"Classification Accuracy: {accuracy:.4f}")


print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=label_encoder.classes_
    )
)


cm = confusion_matrix(
    y_true,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8,7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot(
    ax=ax,
    cmap='Blues',
    values_format='d'
)

plt.title(
    "Enhanced vs Original Morphology Classification"
)

plt.tight_layout()
plt.show()

print("CLASS DISTRIBUTION COMPARISON")

orig_counts = (
    final_pred_df_orig["morphology_label"]
    .value_counts()
)

enh_counts = (
    final_pred_df["morphology_label"]
    .value_counts()
)

comparison_df = np.vstack([
    orig_counts.reindex(label_encoder.classes_, fill_value=0),
    enh_counts.reindex(label_encoder.classes_, fill_value=0)
]).T

for i, cls in enumerate(label_encoder.classes_):

    print(f"\n{cls}")

    print(
        f"Original:  {comparison_df[i][0]}"
    )

    print(
        f"Enhanced: {comparison_df[i][1]}"
    )

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt


final_pred_df_orig["morphology_label"] = (
    final_pred_df_orig.apply(
        classify_galaxy,
        axis=1
    )
)

final_pred_df["morphology_label"] = (
    final_pred_df.apply(
        classify_galaxy,
        axis=1
    )
)

label_encoder = LabelEncoder()

y_labels = label_encoder.fit_transform(
    final_pred_df_orig["morphology_label"]
)

print("Classes:")
print(label_encoder.classes_)


indices = np.arange(len(y_labels))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_labels
)

# ORIGINAL FEATURES
X_train_orig = features[train_idx]
X_test_orig  = features[test_idx]

y_train = y_labels[train_idx]
y_test  = y_labels[test_idx]

# ENHANCED FEATURES
X_train_enh = enhanced_features[train_idx]
X_test_enh  = enhanced_features[test_idx]

clf_orig = LogisticRegression(
    max_iter=5000
)

clf_orig.fit(
    X_train_orig,
    y_train
)

y_pred_orig = clf_orig.predict(
    X_test_orig
)

acc_orig = accuracy_score(
    y_test,
    y_pred_orig
)

clf_enh = LogisticRegression(
    max_iter=5000
)

clf_enh.fit(
    X_train_enh,
    y_train
)

y_pred_enh = clf_enh.predict(
    X_test_enh
)

acc_enh = accuracy_score(
    y_test,
    y_pred_enh
)


print("CLASSIFICATION ACCURACY COMPARISON")
print(f"Original Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:
    print("\n✅ Enhanced images classify better")
else:
    print("\n❌ Original images classify better")

print("ORIGINAL REPORT")
print(
    classification_report(
        y_test,
        y_pred_orig,
        target_names=label_encoder.classes_
    )
)

print("ENHANCED REPORT")
print(
    classification_report(
        y_test,
        y_pred_enh,
        target_names=label_encoder.classes_
    )
)

cm_orig = confusion_matrix(
    y_test,
    y_pred_orig
)

cm_enh = confusion_matrix(
    y_test,
    y_pred_enh
)

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# ORIGINAL
disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm_orig,
    display_labels=label_encoder.classes_
)

disp1.plot(
    ax=axes[0],
    cmap='Blues',
    values_format='d'
)

axes[0].set_title(
    "Original Features"
)

# ENHANCED
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_enh,
    display_labels=label_encoder.classes_
)

disp2.plot(
    ax=axes[1],
    cmap='Blues',
    values_format='d'
)

axes[1].set_title(
    "Enhanced Features"
)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score

bal_acc_orig = balanced_accuracy_score(
    y_test,
    y_pred_orig
)

bal_acc_enh = balanced_accuracy_score(
    y_test,
    y_pred_enh
)

macro_f1_orig = f1_score(
    y_test,
    y_pred_orig,
    average='macro'
)

macro_f1_enh = f1_score(
    y_test,
    y_pred_enh,
    average='macro'
)

print("\nBalanced Accuracy:")
print(f"Original:  {bal_acc_orig:.4f}")
print(f"Enhanced: {bal_acc_enh:.4f}")

print("\nMacro F1:")
print(f"Original:  {macro_f1_orig:.4f}")
print(f"Enhanced: {macro_f1_enh:.4f}")

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import numpy as np

#creating original set labels
final_pred_df_orig["morphology_label"] = (
    final_pred_df_orig.apply(
        classify_galaxy,
        axis=1
    )
)


#creating enhanced set labels
final_pred_df["morphology_label"] = (
    final_pred_df.apply(
        classify_galaxy,
        axis=1
    )
)


label_encoder = LabelEncoder()

label_encoder.fit(
    np.concatenate([
        final_pred_df_orig["morphology_label"],
        final_pred_df["morphology_label"]
    ])
)

# ORIGINAL LABELS
y_orig = label_encoder.transform(
    final_pred_df_orig["morphology_label"]
)

# ENHANCED LABELS
y_enh = label_encoder.transform(
    final_pred_df["morphology_label"]
)

print("Classes:")
print(label_encoder.classes_)

accuracy = accuracy_score(
    y_orig,
    y_enh
)

print("FULL DATASET COMPARISON")

print(f"Classification Agreement Accuracy: {accuracy:.4f}")

print("\nClassification Report:\n")

print(
    classification_report(
        y_orig,
        y_enh,
        target_names=label_encoder.classes_
    )
)


# CONFUSION MATRIX
# ROWS  = ORIGINAL LABELS
# COLS  = ENHANCED LABELS

cm = confusion_matrix(
    y_orig,
    y_enh
)

print("\nConfusion Matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8,7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=label_encoder.classes_
)

disp.plot(
    ax=ax,
    cmap='Blues',
    values_format='d'
)

plt.title(
    "Original vs Enhanced Morphology Labels"
)

plt.tight_layout()
plt.show()

print("CLASS DISTRIBUTION COMPARISON")

orig_counts = (
    final_pred_df_orig["morphology_label"]
    .value_counts()
)

enh_counts = (
    final_pred_df["morphology_label"]
    .value_counts()
)

for cls in label_encoder.classes_:

    print(f"\n{cls}")

    print(
        f"Original:  {orig_counts.get(cls, 0)}"
    )

    print(
        f"Enhanced: {enh_counts.get(cls, 0)}"
    )

In [ ]:
solutions["morphology_label"] = (
    solutions.apply(
        classify_galaxy,
        axis=1
    )
)

print(
    solutions["morphology_label"].value_counts()
)

In [ ]:
ground_truth_df = solutions.copy()

In [ ]:
ground_truth_df.head()

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report
)

from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import numpy as np

ground_truth_df = solutions.copy()

ground_truth_df["morphology_label"] = (
    ground_truth_df.apply(
        classify_galaxy,
        axis=1
    )
)


final_pred_df_orig["predicted_label"] = (
    final_pred_df_orig.apply(
        classify_galaxy,
        axis=1
    )
)


final_pred_df["predicted_label"] = (
    final_pred_df.apply(
        classify_galaxy,
        axis=1
    )
)

#match ground truth with original
orig_eval = final_pred_df_orig.merge(
    ground_truth_df[
        ["GalaxyID", "morphology_label"]
    ],
    on="GalaxyID",
    how="inner",
    suffixes=("_pred", "_true")
)


enh_eval = final_pred_df.merge(
    ground_truth_df[
        ["GalaxyID", "morphology_label"]
    ],
    on="GalaxyID",
    how="inner",
    suffixes=("_pred", "_true")
)


label_encoder = LabelEncoder()

label_encoder.fit(
    ground_truth_df["morphology_label"]
)

# TRUE LABELS
y_true_orig = label_encoder.transform(
    orig_eval["morphology_label_true"]
)

y_true_enh = label_encoder.transform(
    enh_eval["morphology_label_true"]
)

y_pred_orig = label_encoder.transform(
    orig_eval["predicted_label"]
)

y_pred_enh = label_encoder.transform(
    enh_eval["predicted_label"]
)
print("Classes:")
print(label_encoder.classes_)

#accuracy
acc_orig = accuracy_score(
    y_true_orig,
    y_pred_orig
)

acc_enh = accuracy_score(
    y_true_enh,
    y_pred_enh
)

print("CLASSIFICATION ACCURACY")


print(f"Original Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:
    print("\n✅ Enhanced images classify BETTER")
else:
    print("\n❌ Original images classify BETTER")


#original report
print("ORIGINAL REPORT:")
print(
    classification_report(
    y_true_orig,
    y_pred_orig,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
  )
)

#enhanced report
print("ENHANCED REPORT:")
print(
  classification_report(
      y_true_enh,
      y_pred_enh,
      labels=np.arange(len(label_encoder.classes_)),
      target_names=label_encoder.classes_,
      zero_division=0
  )
)

#confusion matrix
cm_orig = confusion_matrix(
    y_true_orig,
    y_pred_orig,
    labels=np.arange(len(label_encoder.classes_))
)

cm_enh = confusion_matrix(
    y_true_enh,
    y_pred_enh,
    labels=np.arange(len(label_encoder.classes_))
)

fig, axes = plt.subplots(1, 2, figsize=(14,6))

# ORIGINAL
disp1 = ConfusionMatrixDisplay(
    confusion_matrix=cm_orig,
    display_labels=label_encoder.classes_
)

disp1.plot(
    ax=axes[0],
    cmap='Blues',
    values_format='d'
)

axes[0].set_title(
    "Original Images"
)

# ENHANCED
disp2 = ConfusionMatrixDisplay(
    confusion_matrix=cm_enh,
    display_labels=label_encoder.classes_
)

disp2.plot(
    ax=axes[1],
    cmap='Blues',
    values_format='d'
)

axes[1].set_title(
    "Enhanced Images"
)

plt.tight_layout()
plt.show()

In [ ]:
print(f" {len(image_df)} images in original dataset")

image_df.head()

In [ ]:
print(f" {len(enhanced_df)} images in enhanced dataset")
enhanced_df.head()

In [ ]:
print(f" {len(solutions)} rows in solution dataset")
solutions.head()

In [ ]:
def classify_galaxy(row):

    # Priority 1: Artifacts / Star
    if row["Class1.3"] > 0.5:
        return "Irregular"

    # Priority 2: Merger
    if row["Class11.1"] > 0.5:
        return "Irregular"

    # Priority 3: Edge-On
    # Class2.2 = Edge-on
    # Class2.1 = Not Edge-on
    if row["Class2.2"] > row["Class2.1"]:
        return "EdgeOn"

    # Priority 4: Morphology
    # Class1.1 = Smooth
    # Class1.2 = Disk/Features
    if row["Class1.1"] >= row["Class1.2"]:
        return "Elliptical"

    else:

        # Disk confirmed
        # Class4.1 = Spiral
        # Class4.2 = Lenticular

        if row["Class4.1"] > row["Class4.2"]:
            return "Spiral"

        else:
            return "Elliptical"

In [ ]:
solutions["morphology_label"] = (
    solutions.apply(
        classify_galaxy,
        axis=1
    )
)

print(
    solutions["morphology_label"].value_counts()
)

In [ ]:
solutions.head()

In [ ]:
edgeon_sample = solutions[solutions['morphology_label'] == 'EdgeOn'].sample(n=1312, random_state=36)
edgeon_sample.head()

In [ ]:
ellip_sample = solutions[solutions['morphology_label'] == 'Elliptical'].sample(n=1312, random_state=36)
ellip_sample.head()


In [ ]:
spiral_sample = solutions[solutions['morphology_label'] == 'Spiral'].sample(n=1312, random_state=36)
spiral_sample.head()

In [ ]:
irreg_sample = solutions[solutions['morphology_label'] == 'Irregular'].copy()
irreg_sample.head()

In [ ]:
classify_sample=pd.concat([ellip_sample,spiral_sample,edgeon_sample,irreg_sample],axis=0,ignore_index=True)
classify_sample.head()

In [ ]:
classify_sample = classify_sample.sample(frac=1, random_state=36).reset_index(drop=True)

In [ ]:
classify_sample.head()

In [ ]:
classify_sample.columns

In [ ]:
enh_classify_sample=enhanced_df.merge(classify_sample,on='GalaxyID',how='inner')
enh_classify_sample=enh_classify_sample.loc[:,['file_loc','GalaxyID']]
enh_classify_sample.head()

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Load your dataset (Replace with your actual file path)
# Let's assume your text column name is 'morphology_class'
df = classify_sample.copy()

# 2. Initialize the LabelEncoder
le = LabelEncoder()

# 3. Fit the encoder and transform the text column into integers
# This creates a new column 'morphology_label' full of 0s, 1s, 2s, and 3s
df['morphology_label'] = le.fit_transform(df['morphology_label'])

# 4. Print out the exact mapping so you can double-check it
print("--- Class Encoding Map ---")
for index, class_name in enumerate(le.classes_):
    print(f"Integer Label {index} ---> Class Name: {class_name}")

print("\nSuccess! Saved encoded dataset.")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier
from PIL import Image

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

# PARALLEL TRAIN/TEST SPLITTING

# Extract the unique IDs and your discrete 4-class target labels
galaxy_ids = df['GalaxyID'].values
labels = df['morphology_label'].values

# 1. First split: Separate out Training (80%) and a temporary block (20%)
train_ids, temp_ids, y_train, y_temp = train_test_split(
    galaxy_ids, labels, test_size=0.20, random_state=42, stratify=labels
)

# 2. Second split: Try to split validation (10%) and test (10%) with stratification.
try:
    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids, y_temp, test_size=0.50, random_state=42, stratify=y_temp
    )
except ValueError:
    print("\n[Warning] One of your galaxy classes has too few samples to stratify twice.")
    print("Falling back to a standard random split for the validation/test sets.")
    # Safe backup split
    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids, y_temp, test_size=0.50, random_state=42
    )

print(f"\nSuccessfully Split Data:")
print(f"Train samples: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

# PYTORCH DATASET ARCHITECTURE
class GalaxyDataset(Dataset):
    def __init__(self, galaxy_ids, labels, image_dir, transform=None):
        self.galaxy_ids = galaxy_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):
        galaxy_id = self.galaxy_ids[idx]

        # Load the image from disk using its GalaxyID
        img_path = os.path.join(self.image_dir, f"{galaxy_id}.jpg")

        # Simple placeholder for image loading logic (using fake tensor if file missing)
        image = Image.open(img_path).convert('RGB')
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return self.transform(image), label

# Instantiate separate dataloaders for original vs enhanced pipelines
ORIG_IMG_DIR = "/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/"
ENHANCED_IMG_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/**/"

train_dataset = GalaxyDataset(train_ids, y_train, ORIG_IMG_DIR, inference_transform)
val_dataset = GalaxyDataset(val_ids, y_val, ORIG_IMG_DIR, enhanced_inference_transform)

# Parallel Test Sets
test_dataset_orig = GalaxyDataset(test_ids, y_test, ORIG_IMG_DIR, inference_transform)
test_dataset_enhanced = GalaxyDataset(test_ids, y_test, ENHANCED_IMG_DIR, enhanced_inference_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader_orig = DataLoader(test_dataset_orig, batch_size=32, shuffle=False)
test_loader_enhanced = DataLoader(test_dataset_enhanced, batch_size=32, shuffle=False)



# INITIALIZE THE ZOOBOT MODEL
# Load the pre-trained ConvNeXT-Nano encoder from Hugging Face via Zoobot
model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    #schema=schemas.gz2_ortho_schema,
    num_classes=3
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)



# MODEL FINE-TUNING LOOP (Baseline)

epochs = 5
print("\n--- Starting Fine-Tuning on Original Training Set ---")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)  # Forward Pass (raw logits output)
        loss = criterion(logits, labels)
        loss.backward()         # Backward Pass
        optimizer.step()        # Update weights

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}] | Training Loss: {epoch_loss:.4f}")



# COMPARATIVE EVALUATION PHASE

def evaluate_model(dataloader, description):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            logits = model(images)  # Forward Pass

            preds = torch.argmax(logits, dim=1)  # Extract final predicted integer class
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f" PERFORMANCE METRICS: {description}")
    print(classification_report(all_labels, all_preds, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

# Evaluate Baseline Performance against Original Test Images
evaluate_model(test_loader_orig, "ORIGINAL BASELINE IMAGES")

# Evaluate Performance against the Enhanced Test Images
evaluate_model(test_loader_enhanced, "ENHANCED IMAGE PIPELINE")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Zoobot specific framework imports
from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier, get_trainer
from galaxy_datasets.pytorch.galaxy_datamodule import CatalogDataModule

torch.manual_seed(42)
np.random.seed(42)


# STEP 1: LOAD DATA AND ENCODE STRING LABELS

# Load dataframe layout
#df = pd.read_csv("galaxy_ground_truth.csv")

df['id_str'] = df['GalaxyID'].astype(str)

# Initialize and apply text encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['morphology_label'])

num_classes = len(le.classes_)
print("--- Class Map Found ---")
for idx, class_name in enumerate(le.classes_):
    print(f"Integer Value {idx} ---> Maps to Class: {class_name}")


# ALIGNED EXPERIMENTAL DATA SPLIT
# Separate out the 80% training set and 20% temporary test/val block
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)

# Split the remaining 20% down the middle into validation and test chunks (10% / 10%)
try:
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=42,
        stratify=temp_df['label']
    )
except ValueError:
    print("\n[Warning] Rare class detected. Splitting validation/test uniformly without stratification.")
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

print(f"\nData successfully split -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


# CREATE DUAL EVALUATION CATALOGS

# Baseline Pipeline DataFrame
test_df_original = test_df.copy()
test_df_original['file_loc'] = test_df_original['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/{x}.jpg")

# Enhanced Pipeline DataFrame (Identical rows, changing image location paths)
test_df_enhanced = test_df.copy()
test_df_enhanced['file_loc'] = test_df_enhanced['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/Galaxy_enhanced/**/{x}.jpg")

# Training / Validation paths setup
train_df['file_loc'] = train_df['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/{x}.jpg")
val_df['file_loc'] = val_df['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/{x}.jpg")

# Convert labels explicitly to standard integers
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)

# STEP 4: INITIALIZE COMPILER DATA MODULES

label_cols = ['label']

# Zoobot uses CatalogDataModule wrapper to handle parallel tensor batch allocations
train_datamodule = CatalogDataModule(
    catalog=train_df,
    label_cols=label_cols,
    batch_size=32,
    num_workers=2
)

# Create standard validation metrics datamodule tracker
val_datamodule = CatalogDataModule(
    catalog=val_df,
    label_cols=label_cols,
    batch_size=32,
    num_workers=2
)

# INITIALIZE & FINE-TUNE ZOOBOT
model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    num_classes=num_classes
)

# Define output file weight save location
save_dir = "/content/mydrive/MyDrive/zoobot_results"
trainer = get_trainer(save_dir=save_dir,max_epochs=5, accelerator="auto")

print("\n--- Initiating Model Training Phase on Baseline Data ---")
trainer.fit(model, datamodule=train_datamodule)

# STEP 6: COMPARATIVE PERFORMANCE INFERENCE
def get_predictions(model, evaluation_catalog):

    # Create temporary isolated Dataloader matching evaluation folders
    eval_datamodule = CatalogDataModule(
        catalog=evaluation_catalog,
        label_cols=label_cols,
        batch_size=32,
        num_workers=2
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    predictions = []
    actuals = []

    with torch.no_grad():
        # Loop directly through the PyTorch data load streams built inside the Datamodule
        for batch in eval_datamodule.test_dataloader():
            images = batch['image'].to(device)
            labels = batch['label']

            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            actuals.extend(labels.numpy())

    return np.array(actuals), np.array(predictions)

# Load the best checked optimization model weights from training
best_checkpoint = trainer.checkpoint_callback.best_model_path
finetuned_model = FinetuneableZoobotClassifier.load_from_checkpoint(best_checkpoint)

# Process inference pass over both datasets
y_true_orig, y_pred_orig = get_predictions(finetuned_model, test_df_original)
y_true_enh, y_pred_enh = get_predictions(finetuned_model, test_df_enhanced)

# STEP 7: PRINT ACCURACY PERFORMANCE COMPARISONS

acc_original = accuracy_score(y_true_orig, y_pred_orig)
acc_enhanced = accuracy_score(y_true_enh, y_pred_enh)

print("\n" + "="*50)
print(f" FINAL EXPERIMENTAL PERFORMANCE SUMMARY")
print("="*50)
print(f"Original Images Classification Accuracy: {acc_original * 100:.2f}%")
print(f"Enhanced Images Classification Accuracy: {acc_enhanced * 100:.2f}%")
print("-"*50)

print("\nDetailed Breakdown for Original Images:")
print(classification_report(y_true_orig, y_pred_orig, target_names=le.classes_))

print("\nDetailed Breakdown for Enhanced Images:")
print(classification_report(y_true_enh, y_pred_enh, target_names=le.classes_))

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Zoobot specific framework imports
from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier, get_trainer
from galaxy_datasets.pytorch.galaxy_datamodule import CatalogDataModule
from zoobot.pytorch.predictions import predict_on_catalog  # <-- CRITICAL HIGHER LEVEL UTILITY

# Ensure exact reproducibility
torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("galaxy_ground_truth.csv")

# Create the required id_str tracking column for the internal galaxy-datasets validation layer
df['id_str'] = df['GalaxyID'].astype(str)

# Initialize text encoder
le = LabelEncoder()
df['label'] = le.fit_transform(df['morphology_class'])

num_classes = len(le.classes_)
print("--- Class Map Found ---")
for idx, class_name in enumerate(le.classes_):
    print(f"Integer Value {idx} ---> Maps to Class: {class_name}")

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=42, stratify=df['label']
)

try:
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
    )
except ValueError:
    print("\n[Warning] Rare class detected. Splitting validation/test uniformly without stratification.")
    val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42)

train_df['file_loc'] = train_df['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/{x}.jpg")
val_df['file_loc'] = val_df['GalaxyID'].apply(lambda x: f"./data/original_images/{x}.jpg")


# INITIALIZE & FINE-TUNE ZOOBOT
train_datamodule = CatalogDataModule(
    catalog=train_df, label_cols=['label'], batch_size=32, num_workers=2
)
val_datamodule = CatalogDataModule(
    catalog=val_df, label_cols=['label'], batch_size=32, num_workers=2
)

model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano', num_classes=num_classes
)

save_dir = "./zoobot_results"
trainer = get_trainer(save_dir=save_dir, max_epochs=5, accelerator="auto")

print("\n--- Initiating Model Training Phase ---")
trainer.fit(model, datamodule=train_datamodule)


# PRODUCTION PREDICTION PIPELINE

# Reload the best performing fine-tuned checkpoint weights
best_checkpoint = trainer.checkpoint_callback.best_model_path
finetuned_model = FinetuneableZoobotClassifier.load_from_checkpoint(best_checkpoint)

# For predictions, Zoobot expects one column name per output probability class
prediction_column_names = [f"{cls_name}_prob" for cls_name in le.classes_]

# Create identical evaluation data tracking targets with shifted path mappings
test_df_original = test_df.copy()
test_df_original['file_loc'] = test_df_original['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/{x}.jpg")

test_df_enhanced = test_df.copy()
test_df_enhanced['file_loc'] = test_df_enhanced['GalaxyID'].apply(lambda x: f"/content/mydrive/MyDrive/Galaxy_enhanced/**/{x}.jpg")

print("\n--- Running Inference Pass on Original Test Catalog ---")
predictions_orig_df = predict_on_catalog.predict(
    catalog=test_df_original,
    model=finetuned_model,
    label_cols=prediction_column_names,
    save_loc=None
)

print("\n--- Running Inference Pass on Enhanced Test Catalog ---")
predictions_enh_df = predict_on_catalog.predict(
    catalog=test_df_enhanced,
    model=finetuned_model,
    label_cols=prediction_column_names,
    save_loc=None
)

y_pred_orig = np.argmax(predictions_orig_df[prediction_column_names].values, axis=1)
y_pred_enh = np.argmax(predictions_enh_df[prediction_column_names].values, axis=1)

y_true = test_df['label'].values

acc_original = accuracy_score(y_true, y_pred_orig)
acc_enhanced = accuracy_score(y_true, y_pred_enh)

print("\n" + "="*50)
print(f" EXPERIMENTAL ACCURACY EVALUATION RESULTS")
print("="*50)
print(f"Original Images Classification Accuracy: {acc_original * 100:.2f}%")
print(f"Enhanced Images Classification Accuracy: {acc_enhanced * 100:.2f}%")
print("="*50)

print("\n[Detailed Multi-Class Breakdown: Original Set]")
print(classification_report(y_true, y_pred_orig, target_names=le.classes_))

print("\n[Detailed Multi-Class Breakdown: Enhanced Set]")
print(classification_report(y_true, y_pred_enh, target_names=le.classes_))

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier

from PIL import Image

import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

galaxy_ids = df['GalaxyID'].values
labels = df['morphology_label'].values

train_ids, temp_ids, y_train, y_temp = train_test_split(
    galaxy_ids,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

try:

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

except ValueError:

    print("\n[Warning] Rare classes detected.")
    print("Using random split for val/test.")

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42
    )

print("\nSuccessfully Split Data:")
print(f"Train: {len(train_ids)}")
print(f"Val:   {len(val_ids)}")
print(f"Test:  {len(test_ids)}")

# LABEL ENCODING

NUM_CLASSES = len(np.unique(labels))

y_train_enc = y_train
y_val_enc   = y_val
y_test_enc  = y_test

print(f"Number of classes: {NUM_CLASSES}")


class GalaxyDataset(Dataset):

    def __init__(
        self,
        galaxy_ids,
        labels,
        image_dir,
        transform=None,
        recursive=False
    ):

        self.galaxy_ids = galaxy_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform
        self.recursive = recursive

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):

        galaxy_id = str(self.galaxy_ids[idx])


        if not self.recursive:

            img_path = os.path.join(
                self.image_dir,
                f"{galaxy_id}.jpg"
            )

        # RECURSIVE SEARCH

        else:

            import glob
            matches = glob.glob(
                os.path.join(
                    self.image_dir,
                    "**",
                    f"{galaxy_id}_conv.jpg"
                ),
                recursive=True
            )

            if len(matches) == 0:

                raise FileNotFoundError(
                    f"Image not found: {galaxy_id}"
                )

            img_path = matches[0]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return image, label


# IMAGE DIRECTORIES

ORIG_IMG_DIR = "/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/"

ENHANCED_IMG_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

train_dataset = GalaxyDataset(
    train_ids,
    y_train_enc,
    ORIG_IMG_DIR,
    inference_transform
)

val_dataset = GalaxyDataset(
    val_ids,
    y_val_enc,
    ORIG_IMG_DIR,
    inference_transform
)

# ORIGINAL TEST SET
test_dataset_orig = GalaxyDataset(
    test_ids,
    y_test_enc,
    ORIG_IMG_DIR,
    inference_transform,
    recursive=False
)

# ENHANCED TEST SET
test_dataset_enhanced = GalaxyDataset(
    test_ids,
    y_test_enc,
    ENHANCED_IMG_DIR,
    enhanced_inference_transform,
    recursive=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader_orig = DataLoader(
    test_dataset_orig,
    batch_size=32,
    shuffle=False
)

test_loader_enhanced = DataLoader(
    test_dataset_enhanced,
    batch_size=32,
    shuffle=False
)

model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    num_classes=NUM_CLASSES
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

epochs = 10

print("\n--- Starting Fine-Tuning ---")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    epoch_loss = (
        running_loss / len(train_loader.dataset)
    )

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Loss: {epoch_loss:.4f}"
    )
    best_loss = float('inf')
    if epoch_loss < best_loss:

      best_loss = epoch_loss

      torch.save(
          model.state_dict(),
          "/content/mydrive/MyDrive/best_zoobot_model.pth"
      )

      print("✅ Best model saved")


# model.load_state_dict(
#     torch.load(
#         "/content/mydrive/MyDrive/best_zoobot_model.pth"
#     )
# )

#print("✅ Saved model loaded")

def evaluate_model(dataloader, description):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)

            logits = model(images)

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    print("\n================================================")
    print(f"{description}")
    print("================================================")

    print(f"\nClassification Accuracy: {accuracy:.4f}")

    print("\nClassification Report:\n")

    print(

        classification_report(
            all_labels,
            all_preds,
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
            digits=4
        )

    )

    # CONFUSION MATRIX

    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=np.arange(NUM_CLASSES)
    )

    print("Confusion Matrix:\n")
    print(cm)

    # PLOT CONFUSION MATRIX

    fig, ax = plt.subplots(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.arange(NUM_CLASSES)
    )

    disp.plot(
        ax=ax,
        cmap='Blues',
        values_format='d'
    )

    plt.title(description)

    plt.tight_layout()

    plt.show()

    return accuracy


acc_orig = evaluate_model(
    test_loader_orig,
    "ORIGINAL TEST SET"
)
acc_enh = evaluate_model(
    test_loader_enhanced,
    "ENHANCED TEST SET"
)

print("\n================================================")
print("FINAL ACCURACY COMPARISON")
print("================================================")

print(f"Original Test Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Test Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:

    print("\n✅ Enhanced images perform BETTER")

else:

    print("\n❌ Original images perform BETTER")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier

from PIL import Image

import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

galaxy_ids = df['GalaxyID'].values
labels = df['morphology_label'].values

train_ids, temp_ids, y_train, y_temp = train_test_split(
    galaxy_ids,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

try:

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

except ValueError:

    print("\n[Warning] Rare classes detected.")
    print("Using random split for val/test.")

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42
    )

print("\nSuccessfully Split Data:")
print(f"Train: {len(train_ids)}")
print(f"Val:   {len(val_ids)}")
print(f"Test:  {len(test_ids)}")

NUM_CLASSES = len(np.unique(labels))

y_train_enc = y_train
y_val_enc   = y_val
y_test_enc  = y_test

print(f"Number of classes: {NUM_CLASSES}")

class GalaxyDataset(Dataset):

    def __init__(
        self,
        galaxy_ids,
        labels,
        image_dir,
        transform=None,
        recursive=False
    ):

        self.galaxy_ids = galaxy_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform
        self.recursive = recursive

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):

        galaxy_id = str(self.galaxy_ids[idx])

        if not self.recursive:

            img_path = os.path.join(
                self.image_dir,
                f"{galaxy_id}.jpg"
            )

        else:

            import glob
            matches = glob.glob(
                os.path.join(
                    self.image_dir,
                    "**",
                    f"{galaxy_id}_conv.jpg"
                ),
                recursive=True
            )

            if len(matches) == 0:

                raise FileNotFoundError(
                    f"Image not found: {galaxy_id}"
                )

            img_path = matches[0]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return image, label

ORIG_IMG_DIR = "/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/"

ENHANCED_IMG_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

train_dataset = GalaxyDataset(
    train_ids,
    y_train_enc,
    ORIG_IMG_DIR,
    inference_transform
)

val_dataset = GalaxyDataset(
    val_ids,
    y_val_enc,
    ORIG_IMG_DIR,
    inference_transform
)

# ORIGINAL TEST SET
test_dataset_orig = GalaxyDataset(
    test_ids,
    y_test_enc,
    ORIG_IMG_DIR,
    inference_transform,
    recursive=False
)

# ENHANCED TEST SET
test_dataset_enhanced = GalaxyDataset(
    test_ids,
    y_test_enc,
    ENHANCED_IMG_DIR,
    enhanced_inference_transform,
    recursive=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader_orig = DataLoader(
    test_dataset_orig,
    batch_size=32,
    shuffle=False
)

test_loader_enhanced = DataLoader(
    test_dataset_enhanced,
    batch_size=32,
    shuffle=False
)

model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    num_classes=NUM_CLASSES
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

# epochs = 10

# print("\n--- Starting Fine-Tuning ---")

# for epoch in range(epochs):

#     model.train()

#     running_loss = 0.0

#     for images, labels in train_loader:

#         images = images.to(device)
#         labels = labels.to(device)

#         optimizer.zero_grad()

#         logits = model(images)

#         loss = criterion(
#             logits,
#             labels
#         )

#         loss.backward()

#         optimizer.step()

#         running_loss += (
#             loss.item() * images.size(0)
#         )

#     epoch_loss = (
#         running_loss / len(train_loader.dataset)
#     )

#     print(
#         f"Epoch [{epoch+1}/{epochs}] "
#         f"| Loss: {epoch_loss:.4f}"
#     )
#     best_loss = float('inf')
#     if epoch_loss < best_loss:

#       best_loss = epoch_loss

#       torch.save(
#           model.state_dict(),
#           "/content/mydrive/MyDrive/best_zoobot_model.pth"
#       )

#       print("✅ Best model saved")

model.load_state_dict(
      torch.load(
          "/content/mydrive/MyDrive/best_zoobot_model.pth",map_location=torch.device(device)
      )
  )

print("✅ Saved model loaded")

def evaluate_model(dataloader, description):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)

            logits = model(images)

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    print("\n================================================")
    print(f"{description}")
    print("================================================")

    print(f"\nClassification Accuracy: {accuracy:.4f}")

    print("\nClassification Report:\n")

    print(

        classification_report(
            all_labels,
            all_preds,
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
            digits=4
        )

    )

    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=np.arange(NUM_CLASSES)
    )

    print("Confusion Matrix:\n")
    print(cm)


    fig, ax = plt.subplots(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.arange(NUM_CLASSES)
    )

    disp.plot(
        ax=ax,
        cmap='Blues',
        values_format='d'
    )

    plt.title(description)

    plt.tight_layout()

    plt.show()

    return accuracy

acc_orig = evaluate_model(
    test_loader_orig,
    "ORIGINAL TEST SET"
)

acc_enh = evaluate_model(
    test_loader_enhanced,
    "ENHANCED TEST SET"
)


print("\n================================================")
print("FINAL ACCURACY COMPARISON")
print("================================================")

print(f"Original Test Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Test Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:

    print("\n✅ Enhanced images perform BETTER")

else:

    print("\n❌ Original images perform BETTER")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier

from PIL import Image

import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

galaxy_ids = df['GalaxyID'].values
labels = df['morphology_label'].values

train_ids, temp_ids, y_train, y_temp = train_test_split(
    galaxy_ids,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

try:

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

except ValueError:

    print("\n[Warning] Rare classes detected.")
    print("Using random split for val/test.")

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42
    )

print("\nSuccessfully Split Data:")
print(f"Train: {len(train_ids)}")
print(f"Val:   {len(val_ids)}")
print(f"Test:  {len(test_ids)}")

NUM_CLASSES = len(np.unique(labels))

y_train_enc = y_train
y_val_enc   = y_val
y_test_enc  = y_test

print(f"Number of classes: {NUM_CLASSES}")

class GalaxyDataset(Dataset):

    def __init__(
        self,
        galaxy_ids,
        labels,
        image_dir,
        transform=None,
        recursive=False
    ):

        self.galaxy_ids = galaxy_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform
        self.recursive = recursive

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):

        galaxy_id = str(self.galaxy_ids[idx])



        if not self.recursive:

            img_path = os.path.join(
                self.image_dir,
                f"{galaxy_id}.jpg"
            )


        else:

            import glob
            matches = glob.glob(
                os.path.join(
                    self.image_dir,
                    "**",
                    f"{galaxy_id}_conv.jpg"
                ),
                recursive=True
            )

            if len(matches) == 0:

                raise FileNotFoundError(
                    f"Image not found: {galaxy_id}"
                )

            img_path = matches[0]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return image, label

ORIG_IMG_DIR = "/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/"

ENHANCED_IMG_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

train_dataset = GalaxyDataset(
    train_ids,
    y_train_enc,
    ORIG_IMG_DIR,
    inference_transform
)

val_dataset = GalaxyDataset(
    val_ids,
    y_val_enc,
    ORIG_IMG_DIR,
    inference_transform
)

# ORIGINAL TEST SET
test_dataset_orig = GalaxyDataset(
    test_ids,
    y_test_enc,
    ORIG_IMG_DIR,
    inference_transform,
    recursive=False
)

# ENHANCED TEST SET
test_dataset_enhanced = GalaxyDataset(
    test_ids,
    y_test_enc,
    ENHANCED_IMG_DIR,
    enhanced_inference_transform,
    recursive=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader_orig = DataLoader(
    test_dataset_orig,
    batch_size=32,
    shuffle=False
)

test_loader_enhanced = DataLoader(
    test_dataset_enhanced,
    batch_size=32,
    shuffle=False
)

model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    num_classes=NUM_CLASSES
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)
epochs = 10

print("\n--- Starting Fine-Tuning ---")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item() * images.size(0)
        )

    epoch_loss = (
        running_loss / len(train_loader.dataset)
    )

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"| Loss: {epoch_loss:.4f}"
    )
    best_loss = float('inf')
    if epoch_loss < best_loss:

      best_loss = epoch_loss

      torch.save(
          model.state_dict(),
          "/content/mydrive/MyDrive/best_zoobot_model.pth"
      )

      print("✅ Best model saved")

# model.load_state_dict(
#       torch.load(
#           "/content/mydrive/MyDrive/best_zoobot_model.pth",map_location=torch.device(device)
#       )
#   )

#print("✅ Saved model loaded")

def evaluate_model(dataloader, description):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)

            logits = model(images)

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    print("\n================================================")
    print(f"{description}")
    print("================================================")

    print(f"\nClassification Accuracy: {accuracy:.4f}")


    print("\nClassification Report:\n")

    print(

        classification_report(
            all_labels,
            all_preds,
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
            digits=4
        )

    )


    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=np.arange(NUM_CLASSES)
    )

    print("Confusion Matrix:\n")
    print(cm)

    fig, ax = plt.subplots(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.arange(NUM_CLASSES)
    )

    disp.plot(
        ax=ax,
        cmap='Blues',
        values_format='d'
    )

    plt.title(description)

    plt.tight_layout()

    plt.show()

    return accuracy

acc_orig = evaluate_model(
    test_loader_orig,
    "ORIGINAL TEST SET"
)

acc_enh = evaluate_model(
    test_loader_enhanced,
    "ENHANCED TEST SET"
)

print("\n================================================")
print("FINAL ACCURACY COMPARISON")
print("================================================")

print(f"Original Test Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Test Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:

    print("\n✅ Enhanced images perform BETTER")

else:

    print("\n❌ Original images perform BETTER")

In [ ]:
classify_sample.to_csv("/content/mydrive/MyDrive/classify_sample.csv", index=False)

In [ ]:
print(type(labels[0]))
print(labels[:5])


In [ ]:
import glob
import os

all_enhanced = glob.glob(
    "/content/mydrive/MyDrive/Galaxy_enhanced/**/*.*",
    recursive=True
)

print("Total enhanced images:", len(all_enhanced))

# Print few examples
for x in all_enhanced[:10]:
    print(os.path.basename(x))

In [ ]:
ENHANCED_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

enhanced_map = {}

all_enhanced = glob.glob(
    os.path.join(
        ENHANCED_DIR,
        "**",
        "*_conv.jpg"
    ),
    recursive=True
)

for path in all_enhanced:

    filename = os.path.basename(path)

    galaxy_id = filename.replace("_conv.jpg", "")

    enhanced_map[galaxy_id] = path

print(f"Indexed {len(enhanced_map)} enhanced images")

In [ ]:
classify_sample_loc=classify_sample.copy()
classify_sample_loc=classify_sample_loc.merge(image_df, on='GalaxyID', how='inner')
classify_sample_loc.head()

In [ ]:
classify_sample["GalaxyID"] = (
    classify_sample["GalaxyID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

image_df["GalaxyID"] = (
    image_df["GalaxyID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

image_paths = image_df[
    ["GalaxyID", "file_loc"]
]

classify_sample = classify_sample.merge(
    image_paths,
    on="GalaxyID",
    how="inner"
)


print(classify_sample.head())

print("\nTotal matched images:",
      len(classify_sample))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import math

batch_size = 25          # images per batch
batch_num = 1
path_col = 'file_loc'


start = batch_num * batch_size
end = min(start + batch_size, len(classify_sample_loc))

batch_df = classify_sample_loc.iloc[start:end]

cols = 5
rows = math.ceil(len(batch_df) / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, 15))

axes = axes.flatten()

for ax, (_, row) in zip(axes, batch_df.iterrows()):
    img = Image.open(row[path_col])

    ax.imshow(img, cmap='gray')
    ax.set_title(f"Idx: {row.name}", fontsize=8)
    ax.axis('off')

# hide unused boxes
for ax in axes[len(batch_df):]:
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"Showing images {start} to {end-1} out of {len(classify_sample_loc)}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import math
import glob
import os

batch_size = 25
batch_num = 0

orig_path_col = 'file_loc'

ENHANCED_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

enhanced_map = {}

all_enhanced = glob.glob(
    os.path.join(
        ENHANCED_DIR,
        "**",
        "*_conv.jpg"
    ),
    recursive=True
)

for path in all_enhanced:

    filename = os.path.basename(path)

    galaxy_id = filename.replace("_conv.jpg", "")

    enhanced_map[galaxy_id] = path

print(f"Indexed {len(enhanced_map)} enhanced images")

start = batch_num * batch_size
end = min(start + batch_size, len(classify_sample_loc))

batch_df = classify_sample_loc.iloc[start:end]

cols = 5
rows = math.ceil(len(batch_df))

fig, axes = plt.subplots(
    rows,
    cols * 2,
    figsize=(20, rows * 4)
)

# Handle single-row case
if rows == 1:
    axes = axes.reshape(1, -1)

for row_idx, (_, row) in enumerate(batch_df.iterrows()):

    galaxy_id = str(row["GalaxyID"])

    orig_img = Image.open(
        row[orig_path_col]
    ).convert('RGB')

    ax_orig = axes[row_idx, 0]

    ax_orig.imshow(orig_img)

    ax_orig.set_title(
        f"Original\nID: {galaxy_id}",
        fontsize=8
    )

    ax_orig.axis('off')

    if galaxy_id in enhanced_map:

        enh_img = Image.open(
            enhanced_map[galaxy_id]
        ).convert('RGB')

        ax_enh = axes[row_idx, 1]

        ax_enh.imshow(enh_img)

        ax_enh.set_title(
            f"Enhanced\nID: {galaxy_id}",
            fontsize=8
        )

        ax_enh.axis('off')

    else:

        ax_enh = axes[row_idx, 1]

        ax_enh.text(
            0.5,
            0.5,
            "Not Found",
            ha='center',
            va='center'
        )

        ax_enh.axis('off')

for i in range(len(batch_df), rows):

    for j in range(cols * 2):

        axes[i, j].axis('off')

plt.tight_layout()

plt.show()

print(
    f"Showing galaxies {start} to {end-1}"
)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Copy dataframe
binary_df = classify_sample.copy()

# Convert to binary classes:
# EdgeOn vs NonEdgeOn

binary_df["edge_binary_class"] = binary_df[
    "morphology_label"
].apply(
    lambda x: "EdgeOn" if x == "EdgeOn" else "NonEdgeOn"
)

le_binary = LabelEncoder()

binary_df["edge_binary_label"] = le_binary.fit_transform(
    binary_df["edge_binary_class"]
)

print("--- Binary Encoding Map ---")

for index, class_name in enumerate(le_binary.classes_):

    print(
        f"Integer Label {index} ---> {class_name}"
    )

print("\nClass Distribution:")

print(
    binary_df["edge_binary_class"].value_counts()
)

binary_df[
    [
        "GalaxyID",
        "morphology_label",
        "edge_binary_class",
        "edge_binary_label"
    ]
].head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
# Convert to binary classes:
# EdgeOn vs NonEdgeOn

binary_df["edge_binary_class"] = binary_df[
    "morphology_label"
].apply(
    lambda x: "EdgeOn" if x == "EdgeOn" else "NonEdgeOn"
)


le_binary = LabelEncoder()

binary_df["edge_binary_label"] = le_binary.fit_transform(
    binary_df["edge_binary_class"]
)


print("--- Binary Encoding Map ---")

for index, class_name in enumerate(le_binary.classes_):

    print(
        f"Integer Label {index} ---> {class_name}"
    )

print("\nClass Distribution:")

print(
    binary_df["edge_binary_class"].value_counts()
)


binary_df[
    [
        "GalaxyID",
        "morphology_label",
        "edge_binary_class",
        "edge_binary_label"
    ]
].head()

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay
)

from zoobot.pytorch.training.finetune import FinetuneableZoobotClassifier

from PIL import Image

import matplotlib.pyplot as plt


torch.manual_seed(42)
np.random.seed(42)

df=binary_df.copy()
galaxy_ids = df['GalaxyID'].values
labels = df['edge_binary_label'].values

train_ids, temp_ids, y_train, y_temp = train_test_split(
    galaxy_ids,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

try:

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42,
        stratify=y_temp
    )

except ValueError:

    print("\n[Warning] Rare classes detected.")
    print("Using random split for val/test.")

    val_ids, test_ids, y_val, y_test = train_test_split(
        temp_ids,
        y_temp,
        test_size=0.50,
        random_state=42
    )

print("\nSuccessfully Split Data:")
print(f"Train: {len(train_ids)}")
print(f"Val:   {len(val_ids)}")
print(f"Test:  {len(test_ids)}")

NUM_CLASSES = len(np.unique(labels))

y_train_enc = y_train
y_val_enc   = y_val
y_test_enc  = y_test

print(f"Number of classes: {NUM_CLASSES}")

class GalaxyDataset(Dataset):

    def __init__(
        self,
        galaxy_ids,
        labels,
        image_dir,
        transform=None,
        recursive=False
    ):

        self.galaxy_ids = galaxy_ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform
        self.recursive = recursive

    def __len__(self):
        return len(self.galaxy_ids)

    def __getitem__(self, idx):

        galaxy_id = str(self.galaxy_ids[idx])

        if not self.recursive:

            img_path = os.path.join(
                self.image_dir,
                f"{galaxy_id}.jpg"
            )


        else:

            import glob
            matches = glob.glob(
                os.path.join(
                    self.image_dir,
                    "**",
                    f"{galaxy_id}_conv.jpg"
                ),
                recursive=True
            )

            if len(matches) == 0:

                raise FileNotFoundError(
                    f"Image not found: {galaxy_id}"
                )

            img_path = matches[0]

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return image, label


ORIG_IMG_DIR = "/content/mydrive/MyDrive/galaxy_zoo/images_training_rev1/images_training_rev1/"

ENHANCED_IMG_DIR = "/content/mydrive/MyDrive/Galaxy_enhanced/"

train_dataset = GalaxyDataset(
    train_ids,
    y_train_enc,
    ORIG_IMG_DIR,
    inference_transform
)

val_dataset = GalaxyDataset(
    val_ids,
    y_val_enc,
    ORIG_IMG_DIR,
    inference_transform
)

# ORIGINAL TEST SET
test_dataset_orig = GalaxyDataset(
    test_ids,
    y_test_enc,
    ORIG_IMG_DIR,
    inference_transform,
    recursive=False
)

# ENHANCED TEST SET
test_dataset_enhanced = GalaxyDataset(
    test_ids,
    y_test_enc,
    ENHANCED_IMG_DIR,
    enhanced_inference_transform,
    recursive=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader_orig = DataLoader(
    test_dataset_orig,
    batch_size=32,
    shuffle=False
)

test_loader_enhanced = DataLoader(
    test_dataset_enhanced,
    batch_size=32,
    shuffle=False
)

model = FinetuneableZoobotClassifier(
    name='hf_hub:mwalmsley/zoobot-encoder-convnext_nano',
    num_classes=NUM_CLASSES
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4
)

# epochs = 10

# print("\n--- Starting Fine-Tuning ---")

# for epoch in range(epochs):

#     model.train()

#     running_loss = 0.0

#     for images, labels in train_loader:

#         images = images.to(device)
#         labels = labels.to(device)

#         optimizer.zero_grad()

#         logits = model(images)

#         loss = criterion(
#             logits,
#             labels
#         )

#         loss.backward()

#         optimizer.step()

#         running_loss += (
#             loss.item() * images.size(0)
#         )

#     epoch_loss = (
#         running_loss / len(train_loader.dataset)
#     )

#     print(
#         f"Epoch [{epoch+1}/{epochs}] "
#         f"| Loss: {epoch_loss:.4f}"
#     )
#     best_loss = float('inf')
#     if epoch_loss < best_loss:

#       best_loss = epoch_loss

#       torch.save(
#           model.state_dict(),
#           "/content/mydrive/MyDrive/best_zoobot_model_edge_nonedge.pth"
#       )

#       print("✅ Best model saved")

model.load_state_dict(
      torch.load(
          "/content/mydrive/MyDrive/best_zoobot_model_edge_nonedge.pth",map_location=torch.device(device)
      )
  )

print("✅ Saved model loaded")

def evaluate_model(dataloader, description):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)

            logits = model(images)

            preds = torch.argmax(
                logits,
                dim=1
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    print("\n================================================")
    print(f"{description}")
    print("================================================")

    print(f"\nClassification Accuracy: {accuracy:.4f}")

    print("\nClassification Report:\n")

    print(

        classification_report(
            all_labels,
            all_preds,
            labels=np.arange(NUM_CLASSES),
            zero_division=0,
            digits=4
        )

    )

    # CONFUSION MATRIX
    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=np.arange(NUM_CLASSES)
    )

    print("Confusion Matrix:\n")
    print(cm)

    fig, ax = plt.subplots(figsize=(7,6))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.arange(NUM_CLASSES)
    )

    disp.plot(
        ax=ax,
        cmap='Blues',
        values_format='d'
    )

    plt.title(description)

    plt.tight_layout()

    plt.show()

    return accuracy



acc_orig = evaluate_model(
    test_loader_orig,
    "ORIGINAL TEST SET"
)



# acc_enh = evaluate_model(
#     test_loader_enhanced,
#     "ENHANCED TEST SET"
# )

# print("\n================================================")
# print("FINAL ACCURACY COMPARISON")
# print("================================================")

# print(f"Original Test Accuracy:  {acc_orig:.4f}")
# print(f"Enhanced Test Accuracy: {acc_enh:.4f}")

# if acc_enh > acc_orig:

#     print("\n✅ Enhanced images perform BETTER")

# else:

#     print("\n❌ Original images perform BETTER")

In [ ]:

acc_enh = evaluate_model(
    test_loader_enhanced,
    "ENHANCED TEST SET"
)

print("\n================================================")
print("FINAL ACCURACY COMPARISON")
print("================================================")

print(f"Original Test Accuracy:  {acc_orig:.4f}")
print(f"Enhanced Test Accuracy: {acc_enh:.4f}")

if acc_enh > acc_orig:

    print("\n✅ Enhanced images perform BETTER")

else:

    print("\n❌ Original images perform BETTER")

In [ ]:
# Get all NonEdgeOn galaxies
non_edge_df = binary_df[
    binary_df["edge_binary_class"] == "NonEdgeOn"
]

# Randomly sample 1312 galaxies
non_edge_sample = non_edge_df.sample(
    n=1312,
    random_state=42
)

print(len(non_edge_sample))

In [ ]:
binary_df = pd.concat([edgeon_sample,non_edge_sample],axis=0,ignore_index=True)

In [ ]:
binary_df.describe()

In [ ]:
binary_df.head()

In [ ]:

classify_sample["GalaxyID"] = (
    classify_sample["GalaxyID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

enh_classify_sample["GalaxyID"] = (
    enh_classify_sample["GalaxyID"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

# FAST LOOKUP DICTIONARIES
orig_map = dict(
    zip(
        classify_sample["GalaxyID"],
        classify_sample["file_loc"]
    )
)

enh_map = dict(
    zip(
        enh_classify_sample["GalaxyID"],
        enh_classify_sample["file_loc"]
    )
)

print("Original images indexed:", len(orig_map))
print("Enhanced images indexed:", len(enh_map))

In [ ]:
import cv2
import numpy as np
from PIL import Image
from skimage.measure import label, regionprops

SAMPLES_PER_CLASS = 100

classes_to_use = [
    "EdgeOn",
    "Elliptical",
    "Spiral",
    "Irregular"
]

def load_gray(path):

    img = Image.open(path).convert('L')

    return np.array(img)

# OTSU STABILITY

def otsu_stability(img):

    _, mask = cv2.threshold(
        img,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    fg = img[mask > 0]
    bg = img[mask == 0]

    if len(fg) == 0 or len(bg) == 0:
        return 0

    score = (
        fg.mean() - bg.mean()
    ) / (bg.std() + 1e-8)

    return score



# FISHER SEPARABILITY

def fisher_separability(img):

    _, mask = cv2.threshold(
        img,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    fg = img[mask > 0]
    bg = img[mask == 0]

    if len(fg) == 0 or len(bg) == 0:
        return 0

    fisher = (
        (fg.mean() - bg.mean())**2
    ) / (
        fg.var() + bg.var() + 1e-8
    )

    return fisher

# CONNECTED COMPONENTS
def connected_components_score(img):

    _, mask = cv2.threshold(
        img,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    num_labels, labels = cv2.connectedComponents(mask)

    return num_labels - 1


#EDGE CONTINUITY
def edge_continuity(img):

    edges = cv2.Canny(img, 50, 150)

    labeled = label(edges)

    props = regionprops(labeled)

    if len(props) == 0:
        return 0

    lengths = [p.area for p in props]

    return np.mean(lengths)


# DICE + IOU
def load_gray_resized(path, size=(256,256)):

    img = Image.open(path).convert('L')

    img = img.resize(size)

    return np.array(img)

def dice_iou(orig_img, enh_img):

    _, mask1 = cv2.threshold(
        orig_img,
        0,
        1,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    _, mask2 = cv2.threshold(
        enh_img,
        0,
        1,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    intersection = np.sum(mask1 * mask2)

    dice = (
        2 * intersection
    ) / (
        np.sum(mask1) + np.sum(mask2) + 1e-8
    )

    iou = (
        intersection
    ) / (
        np.sum((mask1 + mask2) > 0) + 1e-8
    )

    return dice, iou

results = {}

for morph_class in classes_to_use:

    print(f"\n========== {morph_class} ==========")

    # Balanced sampling

    class_df = classify_sample[
        classify_sample["morphology_label"] == morph_class
    ].sample(
        n=SAMPLES_PER_CLASS,
        random_state=42
    )

    orig_otsu = []
    enh_otsu = []

    orig_fisher = []
    enh_fisher = []

    orig_cc = []
    enh_cc = []

    orig_edge = []
    enh_edge = []

    dice_scores = []
    iou_scores = []


    for _, row in class_df.iterrows():

        # galaxy_id = row["GalaxyID"]

        # # ORIGINAL PATH
        # orig_path = sample[sample["GalaxyID"] == galaxy_id
        # ]["file_loc"].values[0]

        # # ENHANCED PATH
        # enh_path = enhanced_sample[
        #     enhanced_sample["GalaxyID"] == galaxy_id
        # ]["file_loc"].values[0]
        galaxy_id = str(row["GalaxyID"])

        if galaxy_id not in orig_map:
            continue

        if galaxy_id not in enh_map:
            continue

        orig_path = orig_map[galaxy_id]
        enh_path  = enh_map[galaxy_id]
        # LOAD
        orig_img = load_gray_resized(orig_path)

        enh_img = load_gray_resized(
            enh_path,
            size=orig_img.shape[::-1]
        )

        # OTSU

        orig_otsu.append(
            otsu_stability(orig_img)
        )

        enh_otsu.append(
            otsu_stability(enh_img)
        )

        # FISHER

        orig_fisher.append(
            fisher_separability(orig_img)
        )

        enh_fisher.append(
            fisher_separability(enh_img)
        )

        # CONNECTED COMPONENTS

        orig_cc.append(
            connected_components_score(orig_img)
        )

        enh_cc.append(
            connected_components_score(enh_img)
        )

        # EDGE CONTINUITY

        orig_edge.append(
            edge_continuity(orig_img)
        )

        enh_edge.append(
            edge_continuity(enh_img)
        )
        # DICE / IOU

        dice, iou = dice_iou(
            orig_img,
            enh_img
        )

        dice_scores.append(dice)
        iou_scores.append(iou)

    results[morph_class] = {

        "Original_Otsu":
            np.mean(orig_otsu),

        "Enhanced_Otsu":
            np.mean(enh_otsu),

        "Original_Fisher":
            np.mean(orig_fisher),

        "Enhanced_Fisher":
            np.mean(enh_fisher),

        "Original_CC":
            np.mean(orig_cc),

        "Enhanced_CC":
            np.mean(enh_cc),

        "Original_Edge":
            np.mean(orig_edge),

        "Enhanced_Edge":
            np.mean(enh_edge),

        "Dice":
            np.mean(dice_scores),

        "IoU":
            np.mean(iou_scores)
    }

    print("\nOtsu Stability")
    print("Original :", results[morph_class]["Original_Otsu"])
    print("Enhanced :", results[morph_class]["Enhanced_Otsu"])

    print("\nFisher Separability")
    print("Original :", results[morph_class]["Original_Fisher"])
    print("Enhanced :", results[morph_class]["Enhanced_Fisher"])

    print("\nConnected Components")
    print("Original :", results[morph_class]["Original_CC"])
    print("Enhanced :", results[morph_class]["Enhanced_CC"])
    print("Lower is cleaner")

    print("\nEdge Continuity")
    print("Original :", results[morph_class]["Original_Edge"])
    print("Enhanced :", results[morph_class]["Enhanced_Edge"])

    print("\nDice Score")
    print(results[morph_class]["Dice"])

    print("\nIoU Score")
    print(results[morph_class]["IoU"])

In [ ]:
# Load one image and apply transform
common_id = enhanced_sample['GalaxyID'].iloc[0]

orig_path = sample[sample['GalaxyID'] == common_id]['file_loc'].values[0]
enh_path = enhanced_sample[enhanced_sample['GalaxyID'] == common_id]['file_loc'].values[0]

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image

enh_map = dict(
    zip(
        enhanced_sample["GalaxyID"].astype(str),
        enhanced_sample["file_loc"]
    )
)

# LOCALIZED ENHANCEMENT FUNCTION

def localized_enhancement(orig_path, enh_path):

    orig = np.array(
        Image.open(orig_path).convert("RGB")
    )

    enh = np.array(
        Image.open(enh_path).convert("RGB")
    )

    # Resize enhanced image to match original
    enh = cv2.resize(
        enh,
        (orig.shape[1], orig.shape[0])
    )

    gray = cv2.cvtColor(
        orig,
        cv2.COLOR_RGB2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (31,31),
        0
    )

    # Galaxy brighter than background
    thresh = np.percentile(
        blur,
        92
    )

    mask = (blur > thresh).astype(np.uint8)

    # Smooth edges
    mask = cv2.GaussianBlur(
        mask.astype(np.float32),
        (61,61),
        0
    )

    mask = mask / mask.max()

    mask = np.expand_dims(
        mask,
        axis=2
    )


    result = (
        enh.astype(np.float32) * mask +
        orig.astype(np.float32) * (1-mask)
    )

    return np.clip(
        result,
        0,
        255
    ).astype(np.uint8)

SAVE_DIR = "/content/localized_images"
os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

localized_rows = []

for _, row in sample.iterrows():

    galaxy_id = str(row["GalaxyID"])

    if galaxy_id not in enh_map:
        continue

    orig_path = row["file_loc"]
    enh_path = enh_map[galaxy_id]

    loc_img = localized_enhancement(
        orig_path,
        enh_path
    )

    save_path = os.path.join(
        SAVE_DIR,
        f"{galaxy_id}_loc.jpg"
    )

    Image.fromarray(
        loc_img
    ).save(save_path)

    localized_rows.append({
        "GalaxyID": galaxy_id,
        "file_loc": save_path
    })

localized_sample = pd.DataFrame(
    localized_rows
)

print(
    "Localized images:",
    len(localized_sample)
)

localized_sample.head()

In [ ]:
sample['GalaxyID'] = sample['GalaxyID'].astype(str)
localized_sample['GalaxyID'] = localized_sample['GalaxyID'].astype(str)
# Load one image and apply transform
common_id = localized_sample['GalaxyID'].iloc[0]

# Successfully find the exact file paths using the ID mask
orig_path = sample[sample['GalaxyID'] == common_id]['file_loc'].values[0]
enh_path =localized_sample[localized_sample['GalaxyID'] == common_id]['file_loc'].values[0]

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def load_gray(path, size=(256,256)):
    img = Image.open(path).convert('L')
    img = img.resize(size)
    return np.array(img, dtype=np.float32)

def plot_isophotes(orig_path, enh_path):

    orig = load_gray(orig_path)
    enh = load_gray(enh_path)

    fig, ax = plt.subplots(2,2, figsize=(10,10))

    ax[0,0].imshow(orig, cmap='gray')
    ax[0,0].set_title("Original")

    ax[0,1].imshow(enh, cmap='gray')
    ax[0,1].set_title("Enhanced")

    levels_orig = np.linspace(
        orig.min(),
        orig.max(),
        12
    )

    levels_enh = np.linspace(
        enh.min(),
        enh.max(),
        12
    )

    ax[1,0].contour(
        orig,
        levels=levels_orig,
        colors='red'
    )
    ax[1,0].invert_yaxis()
    ax[1,0].set_title("Original Isophotes")

    ax[1,1].contour(
        enh,
        levels=levels_enh,
        colors='red'
    )
    ax[1,1].invert_yaxis()
    ax[1,1].set_title("Enhanced Isophotes")

    plt.tight_layout()
    plt.show()

plot_isophotes(orig_path, enh_path)

In [ ]:
def radial_profile(img):

    y, x = np.indices(img.shape)

    cy = img.shape[0] // 2
    cx = img.shape[1] // 2

    r = np.sqrt(
        (x-cx)**2 +
        (y-cy)**2
    )

    r = r.astype(int)

    tbin = np.bincount(
        r.ravel(),
        img.ravel()
    )

    nr = np.bincount(
        r.ravel()
    )

    radial_mean = tbin / np.maximum(nr,1)

    return radial_mean

def compare_radial_profiles(orig_path, enh_path):

    orig = load_gray(orig_path)
    enh = load_gray(enh_path)

    prof_orig = radial_profile(orig)
    prof_enh = radial_profile(enh)

    plt.figure(figsize=(8,5))

    plt.plot(
        prof_orig,
        label='Original'
    )

    plt.plot(
        prof_enh,
        label='Enhanced'
    )

    plt.xlabel("Radius (pixels)")
    plt.ylabel("Mean Brightness")
    plt.title("Radial Brightness Profile")

    plt.legend()
    plt.grid(True)

    plt.show()

compare_radial_profiles(
    orig_path,
    enh_path
)

In [ ]:
def count_detectable_isophotes(img):

    thresh = np.percentile(
        img,
        np.arange(50,95,5)
    )

    count = 0

    for t in thresh:

        mask = img > t

        if mask.sum() > 50:
            count += 1

    return count

orig = load_gray(orig_path)
enh = load_gray(enh_path)

print(
    "Original Isophote Levels:",
    count_detectable_isophotes(orig)
)

print(
    "Enhanced Isophote Levels:",
    count_detectable_isophotes(enh)
)

In [ ]:
import os
os.makedirs('/content/mydrive/MyDrive/galaxy_images', exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

def radial_brightness_profile(img_array, max_radius=None):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    cy, cx = h//2, w//2
    if max_radius is None:
        max_radius = min(cx, cy)
    profiles = []
    for r in range(max_radius):
        Y, X = np.ogrid[:h, :w]
        dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
        ring_mask = (dist >= r) & (dist < r+1)
        if ring_mask.sum() > 0:
            profiles.append(gray[ring_mask].mean())
        else:
            profiles.append(0)
    return np.array(profiles)

# Run across matched pairs
orig_profiles  = []
enh_profiles   = []

N = 100
for i in range(N):
    orig_path_i = sample.iloc[i]['file_loc']
    enh_path_i  = enhanced_sample.iloc[i]['file_loc']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    orig_profiles.append(radial_brightness_profile(orig_arr))
    enh_profiles.append(radial_brightness_profile(local_arr))

# Trim to same length
min_len = min(min(len(p) for p in orig_profiles),
              min(len(p) for p in enh_profiles))

orig_mean = np.array([p[:min_len] for p in orig_profiles]).mean(axis=0)
enh_mean  = np.array([p[:min_len] for p in enh_profiles]).mean(axis=0)
orig_std  = np.array([p[:min_len] for p in orig_profiles]).std(axis=0)
enh_std   = np.array([p[:min_len] for p in enh_profiles]).std(axis=0)

radii = np.arange(min_len)

# Plot mean profiles with confidence bands
plt.figure(figsize=(12, 6))
plt.plot(radii, orig_mean, color='steelblue', linewidth=2, label='Original')
plt.fill_between(radii,
                 orig_mean - orig_std,
                 orig_mean + orig_std,
                 alpha=0.2, color='steelblue')

plt.plot(radii, enh_mean, color='darkorange', linewidth=2, label='Enhanced (Localized)')
plt.fill_between(radii,
                 enh_mean - enh_std,
                 enh_mean + enh_std,
                 alpha=0.2, color='darkorange')

plt.xlabel('Radius (pixels)', fontsize=12)
plt.ylabel('Mean Brightness', fontsize=12)
plt.title('Average Radial Brightness Profile\nOriginal vs Quantum-Enhanced (n=100 galaxies)',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/galaxy_images/radial_profile_mean.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Key statistics
print("=== RADIAL PROFILE STATISTICS ===")
print(f"Peak brightness — Original:  {orig_mean.max():.2f} at r={orig_mean.argmax()}px")
print(f"Peak brightness — Enhanced:  {enh_mean.max():.2f} at r={enh_mean.argmax()}px")
print(f"Peak improvement:            {((enh_mean.max()-orig_mean.max())/orig_mean.max()*100):.1f}%")

# Profile steepness — how fast brightness drops after peak
orig_peak = orig_mean.argmax()
enh_peak  = enh_mean.argmax()
orig_slope = orig_mean[orig_peak] - orig_mean[min(orig_peak+20, min_len-1)]
enh_slope  = enh_mean[enh_peak]  - enh_mean[min(enh_peak+20,  min_len-1)]
print(f"\nSlope after peak (20px) — Original:  {orig_slope:.2f}")
print(f"Slope after peak (20px) — Enhanced:  {enh_slope:.2f}")
print(f"Enhanced profile is {enh_slope/orig_slope:.2f}x steeper")

In [ ]:
def concentration_index(img_array, r_inner_frac=0.2, r_outer_frac=0.8):
    """
    C = 5 * log10(r_80 / r_20)
    r_80 = radius containing 80% of total light
    r_20 = radius containing 20% of total light
    Higher C = more concentrated (elliptical-like)
    Lower C = more extended (spiral-like)
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    cy, cx = h//2, w//2

    # Total flux
    total_flux = gray.sum()
    if total_flux == 0:
        return 0

    # Compute cumulative flux at each radius
    max_r = min(cx, cy)
    cumulative = []
    for r in range(1, max_r):
        Y, X = np.ogrid[:h, :w]
        dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
        circle_mask = dist <= r
        cumulative.append(gray[circle_mask].sum() / total_flux)

    cumulative = np.array(cumulative)
    radii = np.arange(1, max_r)

    # Find r_20 and r_80
    r20 = radii[np.searchsorted(cumulative, 0.20)]
    r80 = radii[np.searchsorted(cumulative, 0.80)]

    if r20 == 0:
        return 0

    C = 5 * np.log10(r80 / r20)
    return C, r20, r80

In [ ]:
def asymmetry_index(img_array):
    """
    A = sum|I - I_180| / sum|I|
    I_180 = image rotated 180 degrees
    Lower A = more symmetric (elliptical)
    Higher A = less symmetric (merger, irregular)
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # Rotate 180 degrees around center
    h, w = gray.shape
    center = (w//2, h//2)
    M = cv2.getRotationMatrix2D(center, 180, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h))

    # Asymmetry
    numerator   = np.abs(gray - rotated).sum()
    denominator = np.abs(gray).sum() + 1e-8

    return numerator / denominator

In [ ]:
def smoothness_index(img_array, sigma=3):
    """
    S = sum|I - I_smooth| / sum|I|
    Higher S = more clumpy (spiral, star-forming)
    Lower S = smoother (elliptical)
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # Smooth version
    smoothed = cv2.GaussianBlur(gray, (0,0), sigma)

    # Residual
    residual = gray - smoothed

    # Only count positive residuals (bright clumps)
    positive_residual = np.maximum(residual, 0)

    S = positive_residual.sum() / (np.abs(gray).sum() + 1e-8)
    return S

In [ ]:
def gini_coefficient(img_array):
    """
    G = 0 means all pixels equally bright
    G = 1 means all light in one pixel
    Ellipticals: high G (~0.6)
    Spirals: lower G (~0.4)
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    pixels = gray.flatten()
    pixels = np.sort(pixels)  # sort ascending

    n = len(pixels)
    index = np.arange(1, n+1)

    G = (2 * (index * pixels).sum()) / (n * pixels.sum() + 1e-8) - (n+1)/n
    return G

In [ ]:
def m20_index(img_array):
    """
    M20 = log10(sum(Mi) / Mtotal)
    for the brightest 20% of pixels
    Sensitive to off-center bright regions
    Mergers: M20 > -1.1
    Ellipticals: M20 < -1.6
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    cy, cx = h//2, w//2

    # Total second moment
    Y, X = np.meshgrid(np.arange(h), np.arange(w), indexing='ij')
    dist_sq = (X - cx)**2 + (Y - cy)**2

    Mtotal = (gray * dist_sq).sum()

    # Sort pixels by brightness descending
    flat_brightness = gray.flatten()
    flat_dist_sq    = dist_sq.flatten()

    sorted_idx = np.argsort(flat_brightness)[::-1]
    sorted_brightness = flat_brightness[sorted_idx]
    sorted_dist_sq    = flat_dist_sq[sorted_idx]

    # Find top 20% flux threshold
    total_flux = flat_brightness.sum()
    cumulative = np.cumsum(sorted_brightness)
    threshold_idx = np.searchsorted(cumulative, 0.20 * total_flux)

    # M20 of brightest 20%
    M20_numerator = (sorted_brightness[:threshold_idx] *
                     sorted_dist_sq[:threshold_idx]).sum()

    if Mtotal == 0 or M20_numerator == 0:
        return 0

    return np.log10(M20_numerator / Mtotal)

In [ ]:
from scipy import stats

results = {
    'Concentration': {'orig': [], 'enh': []},
    'Asymmetry':     {'orig': [], 'enh': []},
    'Smoothness':    {'orig': [], 'enh': []},
    'Gini':          {'orig': [], 'enh': []},
    'M20':           {'orig': [], 'enh': []},
}

N = 65278
for i in range(N):
    orig_path_i = sample.iloc[i]['file_loc']
    enh_path_i  = enhanced_sample.iloc[i]['file_loc']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    for name, fn in [
        ('Concentration', lambda x: concentration_index(x)[0]),
        ('Asymmetry',     asymmetry_index),
        ('Smoothness',    smoothness_index),
        ('Gini',          gini_coefficient),
        ('M20',           m20_index),
    ]:
        try:
            results[name]['orig'].append(fn(orig_arr))
            results[name]['enh'].append(fn(local_arr))
        except:
            continue

# Print results
print("=== MORPHOLOGICAL METRICS ===\n")
for name in results:
    o = np.array(results[name]['orig'])
    e = np.array(results[name]['enh'])
    t, p = stats.ttest_rel(o, e)
    winner = "Enhanced ✅" if e.mean() > o.mean() else "Original ✅"
    if name == 'Asymmetry':  # lower is better for this
        winner = "Enhanced ✅" if e.mean() < o.mean() else "Original ✅"

    print(f"{name}")
    print(f"  Original: {o.mean():.4f} ± {o.std():.4f}")
    print(f"  Enhanced: {e.mean():.4f} ± {e.std():.4f}")
    print(f"  P-value:  {p:.6f} {'✅ significant' if p<0.05 else '❌'}")
    print(f"  Winner:   {winner}\n")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

orig_gini = np.array(results['Gini']['orig'])
enh_gini  = np.array(results['Gini']['enh'])
orig_m20  = np.array(results['M20']['orig'])
enh_m20   = np.array(results['M20']['enh'])

fig, ax = plt.subplots(figsize=(10, 8))

# Plot individual galaxies
ax.scatter(orig_m20, orig_gini, alpha=0.5, color='steelblue',
           s=40, label='Original', zorder=3)
ax.scatter(enh_m20,  enh_gini,  alpha=0.5, color='darkorange',
           s=40, label='Enhanced (Localized)', zorder=3)

# Connect matched pairs with lines
for o_m, o_g, e_m, e_g in zip(orig_m20, orig_gini, enh_m20, enh_gini):
    ax.plot([o_m, e_m], [o_g, e_g], 'gray', alpha=0.15, linewidth=0.8)

# Plot means
ax.scatter(orig_m20.mean(), orig_gini.mean(), color='steelblue',
           s=200, marker='*', zorder=5, edgecolors='black', linewidth=1.5,
           label=f'Original mean ({orig_m20.mean():.2f}, {orig_gini.mean():.2f})')
ax.scatter(enh_m20.mean(), enh_gini.mean(), color='darkorange',
           s=200, marker='*', zorder=5, edgecolors='black', linewidth=1.5,
           label=f'Enhanced mean ({enh_m20.mean():.2f}, {enh_gini.mean():.2f})')

# Standard Gini-M20 classification boundaries (Lotz et al. 2004)
m20_range = np.linspace(-3, 0, 100)

# Merger boundary: G = -0.14*M20 + 0.33
merger_line = -0.14 * m20_range + 0.33
ax.plot(m20_range, merger_line, 'r--', linewidth=2, label='Merger boundary (Lotz+2004)')

# E/S0/Sa boundary: G = -0.14*M20 + 0.33 shifted
early_line = -0.14 * m20_range + 0.14
ax.plot(m20_range, early_line, 'g--', linewidth=2, label='E/S0 boundary')

# Region labels
ax.text(-2.5, 0.75, 'Mergers\n& Irregulars', fontsize=10,
        color='red', style='italic', alpha=0.7)
ax.text(-2.5, 0.55, 'Ellipticals\n& S0s', fontsize=10,
        color='green', style='italic', alpha=0.7)
ax.text(-1.2, 0.45, 'Spirals', fontsize=10,
        color='purple', style='italic', alpha=0.7)

ax.set_xlabel('M20', fontsize=13)
ax.set_ylabel('Gini Coefficient', fontsize=13)
ax.set_title('Gini-M20 Morphological Diagnostic\nOriginal vs Quantum-Enhanced Galaxy Images (n=100)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 0)
ax.set_ylim(0.3, 1.0)

plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/galaxy_images/gini_m20_diagnostic.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"Original  mean — Gini: {orig_gini.mean():.4f}, M20: {orig_m20.mean():.4f}")
print(f"Enhanced  mean — Gini: {enh_gini.mean():.4f},  M20: {enh_m20.mean():.4f}")

In [ ]:
def above_merger_boundary(gini, m20):
    # Merger boundary: G = -0.14*M20 + 0.33
    boundary_gini = -0.14 * m20 + 0.33
    return gini > boundary_gini

orig_above = np.array([above_merger_boundary(g, m)
                        for g, m in zip(orig_gini, orig_m20)])
enh_above  = np.array([above_merger_boundary(g, m)
                        for g, m in zip(enh_gini, enh_m20)])

print(f"Galaxies above merger boundary:")
print(f"  Original:  {orig_above.sum()}/{len(orig_above)} ({orig_above.mean()*100:.1f}%)")
print(f"  Enhanced:  {enh_above.sum()}/{len(enh_above)} ({enh_above.mean()*100:.1f}%)")

# How many crossed the boundary due to enhancement?
crossed = (~orig_above) & enh_above
print(f"  Crossed boundary after enhancement: {crossed.sum()} galaxies ({crossed.mean()*100:.1f}%)")

In [ ]:
# Find the 23 galaxies that crossed the boundary
crossed_indices = np.where(crossed)[0]
print(f"Indices of 23 reclassified galaxies: {crossed_indices[:10]}...")

# Visual inspection of first 6
fig, axes = plt.subplots(6, 3, figsize=(15, 30))

for row, idx in enumerate(crossed_indices[:6]):
    orig_path_i  = sample.iloc[idx]['file_loc']
    enh_path_i   = enhanced_sample.iloc[idx]['file_loc']
    galaxy_id    = sample.iloc[idx]['GalaxyID']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    # Gini-M20 values for this galaxy
    g_orig = orig_gini[idx]
    m_orig = orig_m20[idx]
    g_enh  = enh_gini[idx]
    m_enh  = enh_m20[idx]

    axes[row, 0].imshow(orig_arr)
    axes[row, 0].set_title(f'Original\nGalaxyID: {galaxy_id}\nG={g_orig:.3f}, M20={m_orig:.3f}',
                            fontsize=9, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(local_arr)
    axes[row, 1].set_title(f'Enhanced\nGalaxyID: {galaxy_id}\nG={g_enh:.3f}, M20={m_enh:.3f}',
                            fontsize=9, fontweight='bold')
    axes[row, 1].axis('off')

    # Gini-M20 mini plot for this galaxy
    axes[row, 2].scatter(m_orig, g_orig, color='steelblue', s=150,
                         zorder=5, label='Original')
    axes[row, 2].scatter(m_enh,  g_enh,  color='darkorange', s=150,
                         zorder=5, label='Enhanced')
    axes[row, 2].annotate('', xy=(m_enh, g_enh), xytext=(m_orig, g_orig),
                           arrowprops=dict(arrowstyle='->', color='gray', lw=2))

    # Draw merger boundary
    m_range = np.linspace(-3, 0, 100)
    axes[row, 2].plot(m_range, -0.14*m_range + 0.33, 'r--',
                      linewidth=1.5, label='Merger boundary')
    axes[row, 2].axhline(y=-0.14*m_orig + 0.33, color='gray',
                          alpha=0.3, linestyle=':')
    axes[row, 2].set_xlabel('M20')
    axes[row, 2].set_ylabel('Gini')
    axes[row, 2].set_title('Gini-M20 Shift', fontsize=9)
    axes[row, 2].legend(fontsize=7)
    axes[row, 2].set_xlim(-3, 0)
    axes[row, 2].set_ylim(0.3, 1.0)

plt.suptitle('23 Galaxies Reclassified by Quantum Enhancement\n'
             'Original (Normal) → Enhanced (Merger/Irregular Candidate)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/galaxy_images/reclassified_galaxies.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fix — convert reclassified IDs to integer
reclassified_ids_int = [int(x) for x in reclassified_ids]

# Verify solutions GalaxyID type
print(f"Solutions GalaxyID dtype: {solutions['GalaxyID'].dtype}")
print(f"Sample solutions IDs: {solutions['GalaxyID'].head().tolist()}")

# Now match
reclassified_solutions = solutions[solutions['GalaxyID'].isin(reclassified_ids_int)]
normal_solutions = solutions[~solutions['GalaxyID'].isin(reclassified_ids_int)]

print(f"\nFound in solutions: {len(reclassified_solutions)} out of {len(reclassified_ids_int)}")

# Check volunteer odd feature votes
print("\n=== VOLUNTEER ODD FEATURE VOTES ===")
print(f"Reclassified 23 — mean 'something odd' vote: {reclassified_solutions['Class8.1'].mean():.4f}")
print(f"Normal galaxies  — mean 'something odd' vote: {normal_solutions['Class8.1'].mean():.4f}")

from scipy import stats
t, p = stats.ttest_ind(
    reclassified_solutions['Class8.1'].dropna(),
    normal_solutions['Class8.1'].dropna()
)
print(f"P-value: {p:.6f} {'✅ significant' if p < 0.05 else '❌ not significant'}")

if p < 0.05:
    if reclassified_solutions['Class8.1'].mean() > normal_solutions['Class8.1'].mean():
        print("\n✅ STRONG FINDING: Reclassified galaxies have significantly higher")
        print("   volunteer odd-feature votes — enhancement found REAL disturbed galaxies!")
    else:
        print("\n⚠️ Reclassified galaxies have LOWER odd votes — likely artefact driven")
else:
    print("\n❌ No significant difference — enhancement reclassification may be artefact")

In [ ]:
# Check why NaN — see what GalaxyIDs are in the reclassified set
print("Reclassified GalaxyIDs:")
print(reclassified_ids)

# Check if they exist in solutions
found = solutions[solutions['GalaxyID'].isin(reclassified_ids)]
print(f"\nFound in solutions: {len(found)} out of {len(reclassified_ids)}")

# Check if Class8.1 exists
print(f"\nClass8.1 in solutions columns: {'Class8.1' in solutions.columns}")
print(f"Solutions columns: {solutions.columns.tolist()}")

In [ ]:
from scipy.optimize import curve_fit
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image

def sersic_profile(r, I_e, r_e, n):
    """Sérsic profile function"""
    # bn approximation
    bn = 2*n - 1/3 + 4/(405*n)
    return I_e * np.exp(-bn * ((r/r_e)**(1/n) - 1))

def fit_sersic(img_array, max_radius=100):
    """
    Fit Sérsic profile to galaxy radial brightness
    Returns: r_e (effective radius), n (Sérsic index), fit quality
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    cy, cx = h//2, w//2

    # Get radial profile
    profile = []
    radii = []
    for r in range(1, max_radius):
        Y, X = np.ogrid[:h, :w]
        dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
        ring = (dist >= r) & (dist < r+1)
        if ring.sum() > 0:
            profile.append(gray[ring].mean())
            radii.append(r)

    radii   = np.array(radii, dtype=float)
    profile = np.array(profile, dtype=float)

    # Normalize
    profile = profile / (profile.max() + 1e-8)

    try:
        # Fit Sérsic profile
        popt, pcov = curve_fit(
            sersic_profile,
            radii, profile,
            p0=[1.0, 20.0, 2.0],      # initial guess: I_e=1, r_e=20px, n=2
            bounds=([0, 1, 0.5], [2, 100, 8]),
            maxfev=5000
        )
        I_e, r_e, n = popt

        # Fit quality — R squared
        predicted = sersic_profile(radii, *popt)
        ss_res = np.sum((profile - predicted)**2)
        ss_tot = np.sum((profile - profile.mean())**2)
        r_squared = 1 - ss_res/ss_tot

        return r_e, n, r_squared, radii, profile, predicted

    except Exception as e:
        return None, None, None, radii, profile, None


# ── Run on matched pairs ──────────────────────────────────────────
orig_re  = []
enh_re   = []
orig_n   = []
enh_n    = []
orig_r2  = []
enh_r2   = []
ring_radii = []   # actual ring position from enhanced image

N = 500
for i in range(N):
    orig_path_i = sample.iloc[i]['file_loc']
    enh_path_i  = enhanced_sample.iloc[i]['file_loc']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    # Fit Sérsic to both
    re_o, n_o, r2_o, _, _, _ = fit_sersic(orig_arr)
    re_e, n_e, r2_e, _, _, _ = fit_sersic(local_arr)

    # Find ring position from enhanced image (peak of radial profile)
    gray_e = cv2.cvtColor(local_arr, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray_e.shape
    cy, cx = h//2, w//2
    profile_vals = []
    for r in range(1, 100):
        Y, X = np.ogrid[:h, :w]
        dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
        ring = (dist >= r) & (dist < r+1)
        if ring.sum() > 0:
            profile_vals.append(gray_e[ring].mean())
        else:
            profile_vals.append(0)
    ring_r = np.argmax(profile_vals) + 1

    if re_o is not None:
        orig_re.append(re_o)
        orig_n.append(n_o)
        orig_r2.append(r2_o)
    if re_e is not None:
        enh_re.append(re_e)
        enh_n.append(n_e)
        enh_r2.append(r2_e)
        ring_radii.append(ring_r)

orig_re  = np.array(orig_re)
enh_re   = np.array(enh_re)
orig_n   = np.array(orig_n)
enh_n    = np.array(enh_n)
ring_radii = np.array(ring_radii)

print("=== SÉRSIC PROFILE ANALYSIS ===\n")
print(f"Effective radius r_e:")
print(f"  Original:  {orig_re.mean():.2f} ± {orig_re.std():.2f} px")
print(f"  Enhanced:  {enh_re.mean():.2f} ± {enh_re.std():.2f} px")
print(f"  Ring position (enhanced): {ring_radii.mean():.2f} ± {ring_radii.std():.2f} px")

print(f"\nSérsic index n:")
print(f"  Original:  {orig_n.mean():.3f} ± {orig_n.std():.3f}")
print(f"  Enhanced:  {enh_n.mean():.3f} ± {enh_n.std():.3f}")
print(f"  n=1 = pure disk, n=4 = pure bulge (de Vaucouleurs)")

print(f"\nFit quality R²:")
print(f"  Original:  {np.array(orig_r2).mean():.4f}")
print(f"  Enhanced:  {np.array(enh_r2).mean():.4f}")

# does ring radius match r_e?
correlation, p_val = stats.pearsonr(ring_radii[:len(enh_re)], enh_re)
print(f"\n=== KEY TEST ===")
print(f"Correlation between ring radius and Sérsic r_e:")
print(f"  r = {correlation:.4f}, p = {p_val:.6f}")
if p_val < 0.05 and correlation > 0.5:
    print("✅ STRONG FINDING: Ring radius correlates with Sérsic effective radius!")
    print("   The ring marks the physical bulge-disk transition boundary.")
else:
    print("❌ Ring radius does not correlate with Sérsic r_e")

In [ ]:
# Show Sérsic fit for one example
idx = 0
orig_path_i = sample.iloc[idx]['file_loc']
enh_path_i  = enhanced_sample.iloc[idx]['file_loc']
orig_arr    = np.array(Image.open(orig_path_i).convert('RGB'))
local_arr   = localized_enhancement(orig_path_i, enh_path_i)

re_o, n_o, r2_o, r_o, p_o, fit_o = fit_sersic(orig_arr)
re_e, n_e, r2_e, r_e, p_e, fit_e = fit_sersic(local_arr)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Original image
axes[0].imshow(orig_arr)
axes[0].axvline(x=orig_arr.shape[1]//2, color='white', alpha=0.3)
axes[0].axhline(y=orig_arr.shape[0]//2, color='white', alpha=0.3)
# Draw circle at r_e
circle = plt.Circle((orig_arr.shape[1]//2, orig_arr.shape[0]//2),
                     re_o, color='yellow', fill=False, linewidth=2)
axes[0].add_patch(circle)
axes[0].set_title(f'Original\nr_e={re_o:.1f}px, n={n_o:.2f}', fontweight='bold')
axes[0].axis('off')

# Enhanced image
axes[1].imshow(local_arr)
circle_e = plt.Circle((local_arr.shape[1]//2, local_arr.shape[0]//2),
                       re_e, color='yellow', fill=False, linewidth=2)
circle_ring = plt.Circle((local_arr.shape[1]//2, local_arr.shape[0]//2),
                          ring_radii[idx], color='red', fill=False,
                          linewidth=2, linestyle='--')
axes[1].add_patch(circle_e)
axes[1].add_patch(circle_ring)
axes[1].set_title(f'Enhanced\nr_e={re_e:.1f}px (yellow), ring={ring_radii[idx]}px (red)',
                  fontweight='bold')
axes[1].axis('off')

# Sérsic profile comparison
axes[2].scatter(r_o, p_o, color='steelblue', s=15, alpha=0.6, label='Original data')
axes[2].scatter(r_e, p_e, color='darkorange', s=15, alpha=0.6, label='Enhanced data')
if fit_o is not None:
    axes[2].plot(r_o, fit_o, 'b-', linewidth=2, label=f'Sérsic fit orig (n={n_o:.2f}, R²={r2_o:.3f})')
if fit_e is not None:
    axes[2].plot(r_e, fit_e, 'r-', linewidth=2, label=f'Sérsic fit enh (n={n_e:.2f}, R²={r2_e:.3f})')
axes[2].axvline(x=re_o, color='blue', linestyle='--', alpha=0.7, label=f'r_e orig={re_o:.1f}px')
axes[2].axvline(x=re_e, color='orange', linestyle='--', alpha=0.7, label=f'r_e enh={re_e:.1f}px')
axes[2].axvline(x=ring_radii[idx], color='red', linestyle=':', linewidth=2,
                label=f'Ring position={ring_radii[idx]}px')
axes[2].set_xlabel('Radius (pixels)')
axes[2].set_ylabel('Normalised Brightness')
axes[2].set_title('Sérsic Profile Fit')
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Sérsic Analysis — GalaxyID: {sample.iloc[idx]["GalaxyID"]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/galaxy_images/sersic_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def find_ring_radius(img_array, min_r=5, max_r=80):
    """
    Find ring position more robustly
    — skip the very center (r<5) to avoid core peak
    — find the secondary peak in the profile
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    cy, cx = h//2, w//2

    profile = []
    radii = list(range(min_r, max_r))
    for r in radii:
        Y, X = np.ogrid[:h, :w]
        dist = np.sqrt((X-cx)**2 + (Y-cy)**2)
        ring = (dist >= r) & (dist < r+1)
        if ring.sum() > 0:
            profile.append(gray[ring].mean())
        else:
            profile.append(0)

    profile = np.array(profile)

    # Smooth profile first to avoid noise peaks
    from scipy.ndimage import gaussian_filter1d
    smoothed = gaussian_filter1d(profile, sigma=3)

    # Find peak in smoothed profile
    peak_idx = np.argmax(smoothed)
    return radii[peak_idx], smoothed, np.array(radii)


# Rerun with fixed ring detection
ring_radii_fixed = []
orig_re_fixed    = []
enh_re_fixed     = []

for i in range(max(100,len(sample))):
    orig_path_i = sample.iloc[i]['file_loc']
    enh_path_i  = enhanced_sample.iloc[i]['file_loc']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    re_o, n_o, r2_o, _, _, _ = fit_sersic(orig_arr)
    re_e, n_e, r2_e, _, _, _ = fit_sersic(local_arr)
    ring_r, _, _ = find_ring_radius(local_arr)

    if re_o is not None and re_e is not None:
        orig_re_fixed.append(re_o)
        enh_re_fixed.append(re_e)
        ring_radii_fixed.append(ring_r)

orig_re_fixed  = np.array(orig_re_fixed)
enh_re_fixed   = np.array(enh_re_fixed)
ring_radii_fixed = np.array(ring_radii_fixed)

print(f"Fixed ring position: {ring_radii_fixed.mean():.2f} ± {ring_radii_fixed.std():.2f} px")
print(f"Original r_e:        {orig_re_fixed.mean():.2f} ± {orig_re_fixed.std():.2f} px")
print(f"Ratio ring/r_e:      {(ring_radii_fixed/orig_re_fixed).mean():.3f}")

# Retest correlation
corr, p = stats.pearsonr(ring_radii_fixed, orig_re_fixed)
print(f"\nCorrelation ring vs r_e: r={corr:.4f}, p={p:.6f}")

# Test ratio consistency — is ring always at ~half r_e?
ratio = ring_radii_fixed / orig_re_fixed
print(f"\nRing/r_e ratio: {ratio.mean():.3f} ± {ratio.std():.3f}")
print(f"Consistent with ring ≈ r_e/2: {abs(ratio.mean() - 0.5) < 0.1}")

In [ ]:
from scipy import stats

# Test if ratio is significantly different from 0.5
ratio = ring_radii_fixed / orig_re_fixed

t_stat, p_val = stats.ttest_1samp(ratio, 0.5)
print(f"=== IS RING AT r_e/2? ===")
print(f"Mean ratio:     {ratio.mean():.4f}")
print(f"Std:            {ratio.std():.4f}")
print(f"95% CI:         ({ratio.mean()-1.96*ratio.std()/np.sqrt(len(ratio)):.4f}, "
      f"{ratio.mean()+1.96*ratio.std()/np.sqrt(len(ratio)):.4f})")
print(f"t-test vs 0.5:  t={t_stat:.4f}, p={p_val:.4f}")

if p_val > 0.05:
    print("✅ Ratio is NOT significantly different from 0.5")
    print("   Ring is statistically consistent with r_e/2")
else:
    print(f"❌ Ratio differs from 0.5 (actual mean={ratio.mean():.3f})")

cv = ratio.std() / ratio.mean()  # coefficient of variation
print(f"\nCoefficient of variation: {cv:.3f}")
print(f"{'✅ Consistent ratio (CV<0.3)' if cv < 0.3 else '⚠️ High variability (CV>0.3)'}")

In [ ]:
from scipy.stats import linregress

slope, intercept, r, p, err = linregress(orig_re_fixed,
                                         ring_radii_fixed)

In [ ]:
def bulge_disk_ratio(img_array, ring_radius):
    """
    Estimate B/D ratio using ring as transition boundary

    Bulge flux  = total flux inside ring radius
    Disk flux   = total flux between ring and 2×ring radius
    B/D         = bulge_flux / disk_flux

    Higher B/D = bulge dominated (elliptical)
    Lower B/D  = disk dominated (spiral)
    """
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY).astype(np.float32)
    h, w = gray.shape
    cy, cx = h//2, w//2

    Y, X = np.ogrid[:h, :w]
    dist = np.sqrt((X-cx)**2 + (Y-cy)**2)

    # Bulge region — inside ring
    bulge_mask = dist <= ring_radius
    bulge_flux = gray[bulge_mask].sum()

    # Disk region — between ring and 2x ring
    disk_mask = (dist > ring_radius) & (dist <= 2*ring_radius)
    disk_flux = gray[disk_mask].sum()

    # Normalize by area
    bulge_area = bulge_mask.sum()
    disk_area  = disk_mask.sum()

    if disk_area == 0 or disk_flux == 0:
        return np.nan

    # Surface brightness ratio (area-normalized)
    bulge_sb = bulge_flux / (bulge_area + 1e-8)
    disk_sb  = disk_flux  / (disk_area  + 1e-8)

    BD_ratio = bulge_sb / (disk_sb + 1e-8)

    return BD_ratio

def classify_by_bd_ratio(bd_ratio):
    """
    Standard B/D classification thresholds
    Based on Simard et al. 2011, Driver et al. 2007
    """
    if bd_ratio > 3.0:
        return 'Elliptical'       # bulge dominated
    elif bd_ratio > 1.5:
        return 'Lenticular-S0'    # significant bulge
    elif bd_ratio > 0.8:
        return 'Early-Spiral'     # Sa/Sb
    elif bd_ratio > 0.3:
        return 'Late-Spiral'      # Sc/Sd
    else:
        return 'Irregular-Disk'   # disk/irregular dominated

In [ ]:
from scipy.ndimage import gaussian_filter1d

orig_bd   = []
enh_bd    = []
orig_class = []
enh_class  = []
ring_pos   = []
galaxy_ids = []

N = 500
for i in range(N):
    orig_path_i = sample.iloc[i]['file_loc']
    enh_path_i  = enhanced_sample.iloc[i]['file_loc']
    gid         = sample.iloc[i]['GalaxyID']

    orig_arr  = np.array(Image.open(orig_path_i).convert('RGB'))
    local_arr = localized_enhancement(orig_path_i, enh_path_i)

    # Get ring position from enhanced image
    ring_r, _, _ = find_ring_radius(local_arr)

    # Calculate B/D for both using same ring boundary
    bd_orig = bulge_disk_ratio(orig_arr,  ring_r)
    bd_enh  = bulge_disk_ratio(local_arr, ring_r)

    if not np.isnan(bd_orig) and not np.isnan(bd_enh):
        orig_bd.append(bd_orig)
        enh_bd.append(bd_enh)
        orig_class.append(classify_by_bd_ratio(bd_orig))
        enh_class.append(classify_by_bd_ratio(bd_enh))
        ring_pos.append(ring_r)
        galaxy_ids.append(gid)

orig_bd    = np.array(orig_bd)
enh_bd     = np.array(enh_bd)
ring_pos   = np.array(ring_pos)

print("=== BULGE-TO-DISK RATIO ANALYSIS ===\n")
print(f"Original  B/D: {orig_bd.mean():.4f} ± {orig_bd.std():.4f}")
print(f"Enhanced  B/D: {enh_bd.mean():.4f} ± {enh_bd.std():.4f}")

t, p = stats.ttest_rel(orig_bd, enh_bd)
print(f"P-value: {p:.6f} {'✅ significant' if p<0.05 else '❌'}")

print("\n=== GALAXY TYPE DISTRIBUTION ===\n")
print("Original images:")
for cls in ['Elliptical','Lenticular-S0','Early-Spiral','Late-Spiral','Irregular-Disk']:
    count = orig_class.count(cls)
    print(f"  {cls:20s}: {count:3d} ({count:.0f}%)")

print("\nEnhanced images:")
for cls in ['Elliptical','Lenticular-S0','Early-Spiral','Late-Spiral','Irregular-Disk']:
    count = enh_class.count(cls)
    print(f"  {cls:20s}: {count:3d} ({count:.0f}%)")

In [ ]:
# Get ground truth morphology from Kaggle solutions
bd_df = pd.DataFrame({
    'GalaxyID':   galaxy_ids,
    'BD_orig':    orig_bd,
    'BD_enh':     enh_bd,
    'Class_orig': orig_class,
    'Class_enh':  enh_class,
    'ring_pos':   ring_pos
})

# Convert GalaxyID to integer in both DataFrames
bd_df['GalaxyID'] = bd_df['GalaxyID'].astype(int)
solutions['GalaxyID'] = solutions['GalaxyID'].astype(int)

bd_merged = bd_df.merge(solutions, on='GalaxyID')
# Merge with solutions
bd_merged = bd_df.merge(solutions, on='GalaxyID')

print("=== CROSS-VALIDATION WITH VOLUNTEER VOTES ===\n")

# Check: do high B/D galaxies have higher smooth vote (Class1.1)?
# Smooth galaxies should be bulge-dominated (high B/D)
corr_smooth, p_smooth = stats.pearsonr(bd_merged['BD_orig'],
                                        bd_merged['Class1.1'])
corr_spiral, p_spiral = stats.pearsonr(bd_merged['BD_orig'],
                                        bd_merged['Class4.1'])

print(f"B/D vs smooth vote (Class1.1):")
print(f"  r={corr_smooth:.4f}, p={p_smooth:.6f}")
print(f"  {'✅ Higher B/D = more smooth votes' if corr_smooth>0 else '❌ Unexpected direction'}")

print(f"\nB/D vs spiral vote (Class4.1):")
print(f"  r={corr_spiral:.4f}, p={p_spiral:.6f}")
print(f"  {'✅ Lower B/D = more spiral votes' if corr_spiral<0 else '❌ Unexpected direction'}")

# Does enhanced B/D correlate BETTER with volunteer votes?
corr_enh_smooth, p_enh_smooth = stats.pearsonr(bd_merged['BD_enh'],
                                                 bd_merged['Class1.1'])
print(f"\nEnhanced B/D vs smooth vote:")
print(f"  r={corr_enh_smooth:.4f}, p={p_enh_smooth:.6f}")
print(f"\nOriginal B/D correlation: {corr_smooth:.4f}")
print(f"Enhanced B/D correlation: {corr_enh_smooth:.4f}")
if abs(corr_enh_smooth) > abs(corr_smooth):
    print("✅ STRONG FINDING: Enhanced B/D correlates BETTER with volunteer votes!")
    print("   Ring-based decomposition is more accurate than original image decomposition")
else:
    print("❌ Original B/D correlates better with volunteer votes")

In [ ]:
from scipy.stats import spearmanr
print(spearmanr(bd_merged['BD_enh'],bd_merged['Class1.1']))

In [ ]:
from scipy.stats import spearmanr
print(spearmanr(bd_merged['BD_enh'],bd_merged['Class4.1']))

In [ ]:
classes = ['Elliptical','Lenticular-S0','Early-Spiral','Late-Spiral','Irregular-Disk']
orig_counts = [orig_class.count(c) for c in classes]
enh_counts  = [enh_class.count(c)  for c in classes]

x = np.arange(len(classes))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
bars1 = axes[0].bar(x - width/2, orig_counts, width,
                    label='Original', color='steelblue', alpha=0.8)
bars2 = axes[0].bar(x + width/2, enh_counts,  width,
                    label='Enhanced', color='darkorange', alpha=0.8)

for bar in bars1:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(int(bar.get_height())), ha='center', fontsize=9)
for bar in bars2:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 str(int(bar.get_height())), ha='center', fontsize=9)

axes[0].set_xticks(x)
axes[0].set_xticklabels(classes, rotation=20, ha='right')
axes[0].set_ylabel('Number of Galaxies')
axes[0].set_title('Galaxy Type Distribution\nB/D Ratio Classification', fontweight='bold')
axes[0].legend()

# B/D scatter — original vs enhanced
axes[1].scatter(orig_bd, enh_bd, alpha=0.5, color='purple', s=40)
axes[1].plot([0, orig_bd.max()], [0, orig_bd.max()],
             'k--', linewidth=1, label='y=x (no change)')

# Colour by galaxy class
colours = {'Elliptical':'red', 'Lenticular-S0':'orange',
           'Early-Spiral':'green', 'Late-Spiral':'blue',
           'Irregular-Disk':'purple'}
for cls, col in colours.items():
    mask = np.array(orig_class) == cls
    if mask.sum() > 0:
        axes[1].scatter(orig_bd[mask], enh_bd[mask],
                       color=col, alpha=0.7, s=50, label=cls)

axes[1].set_xlabel('Original B/D Ratio')
axes[1].set_ylabel('Enhanced B/D Ratio')
axes[1].set_title('B/D Ratio Comparison\nOriginal vs Enhanced', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Bulge-to-Disk Ratio Analysis\nQuantum Enhancement Effect on Galaxy Morphology',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/mydrive/MyDrive/galaxy_images/bulge_disk_analysis.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMean B/D — Original: {orig_bd.mean():.4f}")
print(f"Mean B/D — Enhanced: {enh_bd.mean():.4f}")

In [ ]:
def localized_enhancement(orig_path, enh_path):

    orig = np.array(
        Image.open(orig_path).convert("RGB")
    )

    enh = np.array(
        Image.open(enh_path).convert("RGB")
    )

    # Resize enhanced image to match original
    enh = cv2.resize(
        enh,
        (orig.shape[1], orig.shape[0])
    )

    gray = cv2.cvtColor(
        orig,
        cv2.COLOR_RGB2GRAY
    )

    blur = cv2.GaussianBlur(
        gray,
        (31,31),
        0
    )

    # Galaxy brighter than background
    thresh = np.percentile(
        blur,
        92
    )

    mask = (blur > thresh).astype(np.uint8)

    # Smooth edges
    mask = cv2.GaussianBlur(
        mask.astype(np.float32),
        (61,61),
        0
    )

    mask = mask / mask.max()

    mask = np.expand_dims(
        mask,
        axis=2
    )

    # Blend

    result = (
        enh.astype(np.float32) * mask +
        orig.astype(np.float32) * (1-mask)
    )

    return np.clip(
        result,
        0,
        255
    ).astype(np.uint8)